- 데이터 확인
- 중복 데이터 확인
- 이상 데이터 확인
- 손상된 알약을 가진 이미지가 있을까??
- 이미지마다 알약은 몇개씩 있을까??
- 

## 데이터 로드

JSON 라벨 파일 하나와 연결된 이미지 한 장을 읽어 확인한다.
JSON은 이미지 파일명, 알약 위치(bbox), 종류 ID를 담은 라벨 파일이다.
아래 셀을 위에서부터 실행하면 `img`에 RGB 이미지, `annotations`에 라벨이 저장된다.
전체 데이터 로드는 아니며, 다른 JSON에 나뉜 같은 이미지의 라벨은 아직 합치지 않는다.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np

# 1. 프로젝트 최상위 폴더를 찾는다.
# 현재 폴더부터 상위 폴더까지 확인한다.
# src/config.py와 pyproject.toml이 함께 있는 곳을 프로젝트 루트로 판단한다.
PROJECT_ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "config.py").is_file() and (folder / "pyproject.toml").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("pill-bbox 프로젝트 안에서 노트북을 실행해 주세요.")

# 2. JSON 라벨 폴더와 이미지 폴더의 경로를 만든다.
annotation_dir = PROJECT_ROOT / "data" / "raw" / "train_annotations"
image_dir = PROJECT_ROOT / "data" / "raw" / "train_images"

# 3. 하위 폴더까지 탐색해서 JSON 파일 하나를 선택한다.
annotation_path = next(annotation_dir.rglob("*.json"), None)
if annotation_path is None:
    raise FileNotFoundError(f"JSON 라벨 파일이 없습니다: {annotation_dir}")

# 4. JSON 파일을 읽어 파이썬 딕셔너리로 바꾼다.
with annotation_path.open(encoding="utf-8") as file:
    data = json.load(file)

# 5. JSON에 기록된 첫 번째 이미지의 정보를 가져온다.
image_info = data["images"][0]

# 이미지 폴더 경로와 파일명을 합쳐 실제 이미지 경로를 만든다.
image_path = image_dir / image_info["file_name"]

# 6. 이미지 파일을 바이트 배열로 읽고, 이미지 배열로 변환한다.
# 이 방식은 한글이 포함된 경로에서도 이미지를 읽을 수 있게 한다.
img = cv2.imdecode(
    np.fromfile(image_path, dtype=np.uint8),
    cv2.IMREAD_COLOR,
)

# 파일을 이미지로 해석하지 못했다면 오류를 발생시킨다.
if img is None:
    raise ValueError(f"이미지를 읽을 수 없습니다: {image_path}")

# OpenCV 기본 색상 순서인 BGR을 RGB로 바꾼다.
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 7. JSON의 라벨 중 선택한 이미지에 연결된 것만 모은다.
annotations = [
    annotation for annotation in data["annotations"] if annotation["image_id"] == image_info["id"]
]

# 8. 읽은 결과를 확인한다.
print("이미지:", image_path.name)
print("이미지 크기:", img.shape)

for annotation in annotations:
    print("bbox:", annotation["bbox"])
    print("클래스:", annotation["category_id"])

## 데이터별 파일 수 확인

앞의 데이터 로드 셀을 먼저 실행한다. 하위 폴더를 포함해 이미지와 JSON 파일 수를 센다.
JSON은 라벨 파일이므로 이미지 수와 다를 수 있다. 파일 내용은 읽지 않는다.

In [ ]:
# 이미지 확장자는 대소문자 구분 없이 확인한다.
image_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}

# 학습 이미지와 테스트 이미지 수를 각각 센다.
for folder_name in ("train_images", "test_images"):
    folder = PROJECT_ROOT / "data" / "raw" / folder_name
    if not folder.is_dir():
        raise FileNotFoundError(f"데이터 폴더가 없습니다: {folder}")
    image_count = sum(
        1
        for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    print(f"{folder_name}: 이미지 {image_count:,}개")

# 라벨 JSON 수는 이미지 수와 구분해서 출력한다.
annotation_dir = PROJECT_ROOT / "data" / "raw" / "train_annotations"
if not annotation_dir.is_dir():
    raise FileNotFoundError(f"라벨 폴더가 없습니다: {annotation_dir}")
annotation_count = sum(
    1 for path in annotation_dir.rglob("*") if path.is_file() and path.suffix.lower() == ".json"
)
print(f"train_annotations: JSON {annotation_count:,}개")

## 학습·테스트 이미지 구조 비교
앞의 데이터 로드 및 파일 수 확인 셀을 먼저 실행한다.
폴더, 파일명, 확장자, 해상도, 색상 모드를 비교한다. 결과 전체는 `image_structure`에 저장한다.
파일명 숫자의 의미는 데이터 명세로 확인하기 전까지 추정하지 않는다.

In [ ]:
from collections import Counter

import pandas as pd
from PIL import Image

# 전체 픽셀을 읽지 않고 이미지 헤더에서 크기와 색상 모드를 확인한다.
image_structure_rows = []
image_path_sets = {}
for split_name in ("train_images", "test_images"):
    split_dir = PROJECT_ROOT / "data" / "raw" / split_name
    if not split_dir.is_dir():
        raise FileNotFoundError(f"이미지 폴더가 없습니다: {split_dir}")
    image_paths = sorted(
        path
        for path in split_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    image_path_sets[split_name] = {path.name for path in image_paths}
    print(f"\n[{split_name}]")
    print(f"이미지 수: {len(image_paths):,}장")
    print(f"하위 폴더 수: {sum(path.is_dir() for path in split_dir.rglob('*'))}")
    print(
        "이미지 상위 폴더별 수:",
        dict(Counter(path.parent.relative_to(split_dir).as_posix() for path in image_paths)),
    )
    print("확장자별 수:", dict(Counter(path.suffix.lower() for path in image_paths)))
    print("파일명 예시:", [path.name for path in image_paths[:3]])
    print("숫자로만 된 파일명 수:", sum(path.stem.isdecimal() for path in image_paths))
    print("K-로 시작하는 파일명 수:", sum(path.stem.startswith("K-") for path in image_paths))

    for path in image_paths:
        row = {"구분": split_name, "파일명": path.relative_to(split_dir).as_posix()}
        try:
            with Image.open(path) as image:
                row.update({"너비": image.width, "높이": image.height, "색상 모드": image.mode})
        except (OSError, ValueError) as error:
            row["읽기 오류"] = str(error)
        image_structure_rows.append(row)

    split_rows = [row for row in image_structure_rows if row["구분"] == split_name]
    print(
        "해상도별 수 (너비, 높이):",
        dict(Counter((row["너비"], row["높이"]) for row in split_rows if "읽기 오류" not in row)),
    )
    print(
        "색상 모드별 수:",
        dict(Counter(row["색상 모드"] for row in split_rows if "읽기 오류" not in row)),
    )
    print("헤더 읽기 오류 수:", sum("읽기 오류" in row for row in split_rows))

image_structure = pd.DataFrame(image_structure_rows)
shared_image_names = image_path_sets["train_images"] & image_path_sets["test_images"]
print(f"\ntrain/test에서 동일한 파일명: {len(shared_image_names)}개")
print("파일명이 달라도 같은 사진일 수 있으므로 이미지 내용 중복 여부는 별도 검사해야 한다.")
print("헤더 확인만으로 이미지 전체의 손상 여부나 촬영 조건 차이를 판단할 수는 없다.")

## 사진별 JSON 및 라벨 수 확인
앞의 두 코드 셀을 먼저 실행한다. 전체 JSON을 읽어 파일명으로 묶는다.
라벨 수는 원본 annotation 항목 수이며 실제 알약 수와 같은지는 중복 검사가 필요하다.

In [ ]:
from collections import Counter

import pandas as pd

# 사진별 연결 관계를 확인하기 위해 이번에는 모든 JSON을 하나씩 읽는다.
# 이미지 픽셀은 읽지 않고, 파일명과 라벨 개수만 메모리에 저장한다.
image_json_paths = {}
image_label_counts = Counter()
json_count = 0
for annotation_path in annotation_dir.rglob("*"):
    if not annotation_path.is_file() or annotation_path.suffix.lower() != ".json":
        continue
    with annotation_path.open(encoding="utf-8") as file:
        annotation_data = json.load(file)
    json_count += 1
    image_names = {image["id"]: image["file_name"] for image in annotation_data["images"]}
    for image_name in image_names.values():
        image_json_paths.setdefault(image_name, set()).add(annotation_path)
    for label in annotation_data["annotations"]:
        # JSON 안의 image_id로 연결한다. 서로 다른 JSON의 ID를 직접 합치지 않는다.
        image_name = image_names[label["image_id"]]
        image_label_counts[image_name] += 1

# 실제 사진 파일과 JSON에서 참조한 파일을 비교한다.
image_dir = PROJECT_ROOT / "data" / "raw" / "train_images"
actual_image_names = {
    path.relative_to(image_dir).as_posix()
    for path in image_dir.rglob("*")
    if path.is_file() and path.suffix.lower() in image_extensions
}
referenced_image_names = set(image_json_paths)
missing_image_names = referenced_image_names - actual_image_names
unlabeled_image_names = actual_image_names - referenced_image_names

# 파일명이 같으면 같은 이미지로 묶는다. 라벨 개수는 중복 제거 전 원본 항목 수다.
image_summary = pd.DataFrame(
    [
        {
            "이미지 파일명": image_name,
            "JSON 수": len(image_json_paths.get(image_name, set())),
            "라벨 수(중복 제거 전)": image_label_counts[image_name],
            "이미지 파일 존재": image_name in actual_image_names,
        }
        for image_name in sorted(actual_image_names | referenced_image_names)
    ],
    columns=["이미지 파일명", "JSON 수", "라벨 수(중복 제거 전)", "이미지 파일 존재"],
)

print(f"읽은 JSON: {json_count:,}개")
print(f"JSON에서 참조한 서로 다른 이미지: {len(referenced_image_names):,}장")
print(f"실제 학습 이미지: {len(actual_image_names):,}장")
print(f"JSON은 참조하지만 사진 파일이 없는 이미지: {len(missing_image_names):,}장")
print(f"연결된 JSON이 없는 사진: {len(unlabeled_image_names):,}장")
print(f"전체 라벨 항목: {sum(image_label_counts.values()):,}개 (중복 제거 전)")

print("\n이미지별 연결된 JSON 수 분포:")
print(image_summary["JSON 수"].value_counts().sort_index().rename_axis("JSON 수").to_string())
print("\n사진별 결과 예시 10개 (전체 결과는 image_summary에 저장):")
print(image_summary.head(10).to_string(index=False))

- train_annotations은 알약 종류뿐 아니라 사진 속 위치도 알려주는 정답 데이터
- 데이터는 사진 하나에 JSON이 여러 개 연결되어 있음
- 학습할 때는 같은 사진에 해당하는 라벨들을 모아줘야함

## 선택한 이미지에 연결된 JSON 모두 찾기
데이터 로드 셀을 먼저 실행한다. JSON 내부의 이미지 파일명을 기준으로 전체 라벨 폴더를 검색한다.
연결된 JSON 경로와 제품명, bbox를 출력하고 라벨을 모은다. 아래 시각화 셀은 모은 라벨 전체를 그린다.

In [ ]:
# 앞의 데이터 로드 셀에서 선택한 사진을 기준으로 찾는다.
# 다른 사진을 확인하려면 이 값을 train_images 안의 파일명으로 바꾼다.
target_image_name = image_info["file_name"]
matched_json_paths = []
all_annotations = []

# JSON 파일명 대신 내부 images의 file_name을 비교해야 정확하게 연결할 수 있다.
for json_path in sorted(annotation_dir.rglob("*.json")):
    with json_path.open(encoding="utf-8") as file:
        json_data = json.load(file)

    # image_id는 JSON마다 다를 수 있으므로 현재 JSON 안에서 찾아 연결한다.
    matched_image_ids = {
        image["id"] for image in json_data["images"] if image["file_name"] == target_image_name
    }
    if not matched_image_ids:
        continue

    matched_json_paths.append(json_path)
    matched_annotations = [
        label for label in json_data["annotations"] if label["image_id"] in matched_image_ids
    ]
    category_names = {
        category["id"]: category.get("name", "이름 없음") for category in json_data["categories"]
    }
    all_annotations.extend(matched_annotations)

    print(f"\nJSON: {json_path.relative_to(PROJECT_ROOT)}")
    for label in matched_annotations:
        category_id = label["category_id"]
        print(
            f"  종류 ID: {category_id}, "
            f"제품명: {category_names.get(category_id, '이름 없음')}, "
            f"bbox: {label['bbox']}"
        )

print(f"\n대상 이미지: {target_image_name}")
print(f"연결된 JSON: {len(matched_json_paths)}개")
print(f"모은 라벨: {len(all_annotations)}개 (중복 제거 전)")

# 아래 시각화 셀에서 선택한 사진과 모은 라벨 전체를 사용한다.
image_path = image_dir / target_image_name
img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
if img is None:
    raise ValueError(f"이미지를 읽을 수 없습니다: {image_path}")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
annotations = all_annotations

- 이미지 하나에 JSON이 모두 연결되어 있다는 것을 확인하기 위함

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# JSON에 위치와 종류 정보가 있는지 확인한다.
for annotation in annotations:
    print("위치 정보:", annotation.get("bbox"))
    print("종류 ID:", annotation.get("category_id"))

# 사진 위에 bbox를 그린다.
fig, ax = plt.subplots(figsize=(8, 10))
ax.imshow(img)

for annotation in annotations:
    # bbox를 [왼쪽 x, 위쪽 y, 너비, 높이] 형식으로 해석한다.
    left, top, width, height = annotation["bbox"]

    ax.add_patch(
        Rectangle(
            (left, top),
            width,
            height,
            edgecolor="red",
            facecolor="none",
            linewidth=2,
        )
    )
    ax.text(
        left,
        top,
        str(annotation["category_id"]),
        color="white",
        backgroundcolor="red",
    )

ax.set_title("Annotation bbox")
ax.axis("off")
plt.show()

- 이미지 안에 있는 알약이 JSON이 다 연결되어 있다는걸 시각화를 통해 직접 확인

## JSON 및 라벨 중복 확인
데이터 로드 셀을 먼저 실행한다. 전체 JSON을 읽되 이미지는 읽지 않는다.
파일명 중복, 전체 내용 중복, 같은 이미지·종류·bbox의 반복을 구분한다.
bbox가 조금 다른 유사 라벨은 이 검사로 찾지 않는다. 반복 라벨은 삭제 전 다른 필드도 비교해야 한다.

In [ ]:
import hashlib
from collections import defaultdict

# 파일명이 같다는 이유만으로 중복으로 판단하지 않는다.
# 1. JSON 전체 내용이 같은지 검사한다 (공백과 객체 키 순서는 무시).
# 2. 같은 사진에 동일한 종류 ID와 bbox가 반복되는지 별도로 검사한다.
content_groups = defaultdict(list)
filename_groups = defaultdict(list)
label_groups = defaultdict(list)
checked_json_count = 0

for json_path in sorted(annotation_dir.rglob("*")):
    if not json_path.is_file() or json_path.suffix.lower() != ".json":
        continue
    with json_path.open(encoding="utf-8") as file:
        json_data = json.load(file)
    checked_json_count += 1
    relative_path = json_path.relative_to(PROJECT_ROOT).as_posix()
    filename_groups[json_path.name].append(relative_path)

    # 배열 순서와 모든 필드 값은 유지하여 전체 내용이 같은 파일만 묶는다.
    normalized_json = json.dumps(json_data, sort_keys=True, ensure_ascii=False)
    content_hash = hashlib.sha256(normalized_json.encode("utf-8")).hexdigest()
    content_groups[content_hash].append(relative_path)

    image_names = {image["id"]: image["file_name"] for image in json_data["images"]}
    for label_index, label in enumerate(json_data["annotations"]):
        # JSON마다 달라질 수 있는 라벨 ID 대신 사진·종류·위치로 비교한다.
        label_key = (
            image_names[label["image_id"]],
            label["category_id"],
            tuple(label["bbox"]),
        )
        label_groups[label_key].append(
            {"JSON 경로": relative_path, "라벨 인덱스": label_index, "라벨 ID": label.get("id")}
        )

# 이 변수에는 중복 그룹 전체를 보관한다. 원본 파일은 삭제하거나 수정하지 않는다.
duplicate_json_groups = [paths for paths in content_groups.values() if len(paths) > 1]
duplicate_label_groups = {key: entries for key, entries in label_groups.items() if len(entries) > 1}
same_filename_groups = {name: paths for name, paths in filename_groups.items() if len(paths) > 1}

print(f"검사한 JSON: {checked_json_count:,}개")
print(f"전체 내용이 같은 JSON: {len(duplicate_json_groups):,}그룹")
print(
    "그룹당 하나만 남겼을 때 중복 파일 수: "
    f"{sum(len(paths) - 1 for paths in duplicate_json_groups):,}개"
)
print(f"같은 사진·종류·bbox가 반복된 라벨: {len(duplicate_label_groups):,}그룹")
print(f"파일명만 같은 JSON: {len(same_filename_groups):,}그룹 (내용 중복과 별개)")

# 출력이 길어지지 않도록 각 검사 결과의 첫 5그룹만 보여 준다.
for group_number, paths in enumerate(duplicate_json_groups[:5], start=1):
    print(f"\n내용 중복 그룹 {group_number}:")
    for path in paths:
        print(f"  {path}")

for label_key, entries in list(duplicate_label_groups.items())[:5]:
    print(f"\n반복 라벨: 사진={label_key[0]}, 종류={label_key[1]}, bbox={label_key[2]}")
    for entry in entries:
        print(f"  {entry}")

- 파일명만 같은 JSON = 232그룹 : train_images와 크기가 같음
- JSON 763개가 232그룹안에 여러개씩 나눠서 들어있다고 생각할 수 있음

## 데이터 이상·누락·중복 검사
데이터 로드와 파일 수 확인 셀을 먼저 실행한다. 전체 이미지 픽셀과 JSON을 순차 검사한다.
손실은 파일 읽기 실패·연결 누락·필수 구조 오류로 확인하며, 원본 자료가 없으므로 원래 받을 파일의 유실 여부는 알 수 없다.
전체 오류는 `quality_report`, 동일 픽셀 그룹은 `exact_image_duplicates`, bbox 검토 후보는 `bbox_outlier_candidates`에 저장한다.
이미지 유사 중복과 실제 알약 라벨 누락은 자동 검사 범위에 포함되지 않는다. 원본 파일은 수정하지 않는다.

In [ ]:
import hashlib
import math
from collections import defaultdict

import pandas as pd
from PIL import Image

# 파일을 삭제하지 않고 검사 결과만 저장한다. 이미지는 한 장씩 읽고 메모리에서 해제한다.
quality_issues = []
pixel_groups = defaultdict(list)
perceptual_hashes = []
image_metadata = {}
split_image_counts = {}
for split_name in ("train_images", "test_images"):
    split_dir = PROJECT_ROOT / "data" / "raw" / split_name
    if not split_dir.is_dir():
        raise FileNotFoundError(f"이미지 폴더가 없습니다: {split_dir}")
    image_paths = sorted(
        path
        for path in split_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    split_image_counts[split_name] = len(image_paths)
    for image_path in image_paths:
        name = image_path.relative_to(split_dir).as_posix()
        image_metadata[(split_name, name)] = None
        try:
            # verify로 파일 형식을 검사한 뒤 다시 열어 전체 픽셀을 디코딩한다.
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(image_path) as image:
                image.load()
                image_metadata[(split_name, name)] = image.size
                rgb = image.convert("RGB")
                digest = hashlib.sha256(rgb.tobytes()).hexdigest()
                # 64비트 차이 해시는 유사 후보용이다. 완전 중복 판정에는 사용하지 않는다.
                small = list(rgb.convert("L").resize((9, 8)).getdata())
                difference_hash = 0
                for row_index in range(8):
                    for column_index in range(8):
                        offset = row_index * 9 + column_index
                        difference_hash = (difference_hash << 1) | (
                            small[offset] > small[offset + 1]
                        )
                perceptual_hashes.append((split_name, name, difference_hash, digest))
                pixel_groups[(rgb.size, digest)].append((split_name, name))
        except (OSError, ValueError, SyntaxError) as error:
            quality_issues.append(
                {
                    "구분": split_name,
                    "파일": name,
                    "문제": "이미지 손상/읽기 실패",
                    "상세": str(error),
                }
            )
    print(f"{split_name}: 이미지 {len(image_paths):,}장 검사 완료", flush=True)

# 픽셀이 완전히 같은 사진만 중복으로 잡는다. 회전·재촬영·유사 사진 검사는 별도다.
exact_image_duplicates = [entries for entries in pixel_groups.values() if len(entries) > 1]
cross_split_duplicates = [
    entries for entries in exact_image_duplicates if len({split for split, name in entries}) > 1
]

annotation_dir = PROJECT_ROOT / "data" / "raw" / "train_annotations"
if not annotation_dir.is_dir():
    raise FileNotFoundError(f"라벨 폴더가 없습니다: {annotation_dir}")
referenced_images = set()
labeled_images = set()
json_content_groups = defaultdict(list)
quality_label_groups = defaultdict(list)
bbox_rows = []
json_paths = sorted(
    path for path in annotation_dir.rglob("*") if path.is_file() and path.suffix.lower() == ".json"
)
for json_path in json_paths:
    relative_path = json_path.relative_to(PROJECT_ROOT).as_posix()
    try:
        with json_path.open(encoding="utf-8") as file:
            json_data = json.load(file)
        if not isinstance(json_data, dict) or any(
            not isinstance(json_data.get(key), list)
            for key in ("images", "annotations", "categories")
        ):
            raise ValueError("images, annotations, categories 배열이 필요합니다.")
        normalized = json.dumps(json_data, sort_keys=True, ensure_ascii=False)
        json_content_groups[hashlib.sha256(normalized.encode()).hexdigest()].append(relative_path)
        images_by_id = {}
        for image in json_data["images"]:
            if image["id"] in images_by_id:
                raise ValueError("JSON 내부의 이미지 ID가 중복됩니다.")
            images_by_id[image["id"]] = image
            name = image["file_name"]
            referenced_images.add(name)
            actual_size = image_metadata.get(("train_images", name))
            if ("train_images", name) not in image_metadata:
                quality_issues.append(
                    {
                        "구분": "annotation",
                        "파일": relative_path,
                        "문제": "연결된 이미지 파일 없음",
                        "상세": name,
                    }
                )
            elif actual_size is not None and actual_size != (image["width"], image["height"]):
                quality_issues.append(
                    {
                        "구분": "annotation",
                        "파일": relative_path,
                        "문제": "JSON과 실제 이미지 크기 불일치",
                        "상세": name,
                    }
                )
            if image["width"] <= 0 or image["height"] <= 0:
                raise ValueError("이미지 너비와 높이는 양수여야 합니다.")
        category_ids = {category["id"] for category in json_data["categories"]}
        for label_index, label in enumerate(json_data["annotations"]):
            # 오류가 있는 라벨 하나 때문에 나머지 라벨 검사가 중단되지 않게 한다.
            try:
                image = images_by_id[label["image_id"]]
                name = image["file_name"]
                labeled_images.add(name)
                if label["category_id"] not in category_ids:
                    raise ValueError("category_id에 해당하는 category가 없습니다.")
                bbox = label["bbox"]
                if (
                    not isinstance(bbox, list)
                    or len(bbox) != 4
                    or any(
                        isinstance(value, bool)
                        or not isinstance(value, (int, float))
                        or not math.isfinite(value)
                        for value in bbox
                    )
                ):
                    raise ValueError("bbox는 유한한 숫자 4개여야 합니다.")
                left, top, width, height = bbox
                if width <= 0 or height <= 0:
                    raise ValueError("bbox 너비 또는 높이가 0 이하입니다.")
                if (
                    left < 0
                    or top < 0
                    or left + width > image["width"]
                    or top + height > image["height"]
                ):
                    raise ValueError("bbox가 이미지 경계를 벗어납니다.")
                quality_label_groups[(name, label["category_id"], tuple(bbox))].append(
                    (relative_path, label_index)
                )
                bbox_rows.append(
                    {
                        "이미지": name,
                        "JSON": relative_path,
                        "종류 ID": label["category_id"],
                        "bbox": bbox,
                        "면적 비율": width * height / (image["width"] * image["height"]),
                    }
                )
            except (KeyError, TypeError, ValueError) as error:
                quality_issues.append(
                    {
                        "구분": "annotation",
                        "파일": relative_path,
                        "문제": "라벨 오류",
                        "상세": f"인덱스 {label_index}: {error}",
                    }
                )
    except (OSError, UnicodeError, ValueError, KeyError, TypeError) as error:
        quality_issues.append(
            {
                "구분": "annotation",
                "파일": relative_path,
                "문제": "JSON 파싱/구조 오류",
                "상세": str(error),
            }
        )

train_names = {name for split, name in image_metadata if split == "train_images"}
images_without_json = sorted(train_names - referenced_images)
images_without_labels = sorted(train_names - labeled_images)
duplicate_contents = [paths for paths in json_content_groups.values() if len(paths) > 1]
duplicate_labels = [entries for entries in quality_label_groups.values() if len(entries) > 1]
quality_report = pd.DataFrame(quality_issues, columns=["구분", "파일", "문제", "상세"])

# IQR 기준은 오류 판정이 아니라, 시각적으로 확인할 후보를 추리는 용도다.
bbox_quality = pd.DataFrame(bbox_rows, columns=["이미지", "JSON", "종류 ID", "bbox", "면적 비율"])
bbox_outlier_candidates = bbox_quality.copy()
if not bbox_quality.empty:
    lower_quartile = bbox_quality["면적 비율"].quantile(0.25)
    upper_quartile = bbox_quality["면적 비율"].quantile(0.75)
    spread = upper_quartile - lower_quartile
    bbox_outlier_candidates = bbox_quality[
        (bbox_quality["면적 비율"] < lower_quartile - 1.5 * spread)
        | (bbox_quality["면적 비율"] > upper_quartile + 1.5 * spread)
    ]

print(f"\nJSON 검사: {len(json_paths):,}개")
print("손상/누락/라벨 오류 유형별 수:")
print(
    quality_report["문제"].value_counts().to_string() if not quality_report.empty else "발견 없음"
)
print(f"연결된 JSON이 없는 학습 이미지: {len(images_without_json)}장")
print(f"annotation 항목이 연결되지 않은 학습 이미지: {len(images_without_labels)}장")
print(f"전체 내용이 같은 JSON: {len(duplicate_contents)}그룹")
print(f"같은 사진·종류·bbox의 정상 범위 라벨 중복: {len(duplicate_labels)}그룹")
for split_name in ("train_images", "test_images"):
    within_split = [
        entries
        for entries in exact_image_duplicates
        if sum(split == split_name for split, name in entries) > 1
    ]
    print(f"{split_name} 내부 픽셀 중복: {len(within_split)}그룹")
print(f"train/test 간 픽셀 중복: {len(cross_split_duplicates)}그룹")
print(f"bbox 면적 비율의 통계적 이상치 후보: {len(bbox_outlier_candidates)}개 (오류 확정 아님)")
if not quality_report.empty:
    print("\n오류 예시 10개:")
    print(quality_report.head(10).to_string(index=False))
if exact_image_duplicates:
    print("\n픽셀 중복 예시 5그룹:", exact_image_duplicates[:5])
if not bbox_outlier_candidates.empty:
    print("\n통계적 이상치 후보 예시 5개:")
    print(bbox_outlier_candidates.head(5).to_string(index=False))
print("\n테스트 정답 라벨이 제공되지 않아 테스트 라벨 누락은 판단하지 않는다.")
print("알약 라벨이 실제로 빠졌는지, bbox와 종류가 정확한지는 별도 시각 검토가 필요하다.")

- 확인해야할 데이터가 라벨 오류 1개, 이상치 후보 24개임 이것들을 확인

## 오류 및 이상치 후보 전체 시각화
데이터 이상·누락·중복 검사 셀을 먼저 실행한다. 면적 분포와 통계적 후보 전체를 표시한다.
빨간 사각형이 대상 알약을 잘 감싸는지, 다른 알약이나 배경을 과하게 포함하는지 확인한다.
경계 오류는 초록색 이미지 영역과 빨간색 bbox를 확장 좌표로 비교한다.
통계적 후보는 크기가 특이하다는 의미이며 오류로 확정하지 않는다.

In [ ]:
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# 먼저 '데이터 이상·누락·중복 검사' 셀을 실행한다.
# IQR 기준: 전체 유효 bbox 면적 비율의 중앙 50% 구간을 기준으로 후보를 찾는다.
review_rows = []
if not bbox_quality.empty:
    lower_limit = lower_quartile - 1.5 * spread
    upper_limit = upper_quartile + 1.5 * spread
    print(f"정상 참고 구간: 면적 비율 {lower_limit:.2%} ~ {upper_limit:.2%}")
    print("이 구간 밖이면 통계적 후보이며, 알약을 정확히 감싸면 정상 라벨일 수 있다.")

    # 전체 분포에 후보와 판정 기준선을 함께 표시한다.
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(bbox_quality["면적 비율"] * 100, bins=30, color="steelblue")
    ax.axvline(lower_limit * 100, color="orange", linestyle="--", label="Lower IQR fence")
    ax.axvline(upper_limit * 100, color="red", linestyle="--", label="Upper IQR fence")
    ax.set(
        xlabel="BBox area / image area (%)", ylabel="Label count", title="BBox area distribution"
    )
    ax.legend()
    plt.tight_layout()
    plt.show()

    # 첫 몇 개만 고르지 않고 후보 전체를 4개씩 나누어 표시한다.
    candidates = bbox_outlier_candidates.to_dict("records")
    for start in range(0, len(candidates), 4):
        page = candidates[start : start + 4]
        fig, axes = plt.subplots(2, 2, figsize=(12, 15), squeeze=False)
        for offset, ax in enumerate(axes.flat):
            if offset >= len(page):
                ax.axis("off")
                continue
            row = page[offset]
            number = start + offset + 1
            with Image.open(
                PROJECT_ROOT / "data" / "raw" / "train_images" / row["이미지"]
            ) as image:
                ax.imshow(image.convert("RGB"))
                image_width, image_height = image.size
            left, top, width, height = row["bbox"]
            ax.add_patch(
                Rectangle((left, top), width, height, fill=False, color="red", linewidth=2)
            )
            ax.set_xlim(0, image_width)
            ax.set_ylim(image_height, 0)
            direction = "large" if row["면적 비율"] > upper_limit else "small"
            ax.set_title(
                f"Candidate {number}: unusually {direction} bbox\n"
                f"class={row['종류 ID']}, area={row['면적 비율']:.2%}\n"
                f"bbox={row['bbox']}",
                fontsize=10,
            )
            ax.set_xlabel(row["이미지"], fontsize=7)
            review_rows.append(
                {
                    "후보 번호": number,
                    "이미지": row["이미지"],
                    "종류 ID": row["종류 ID"],
                    "면적 비율(%)": round(row["면적 비율"] * 100, 2),
                    "상한(%)": round(upper_limit * 100, 2),
                    "이유": "면적 비율이 IQR 상한 초과"
                    if direction == "large"
                    else "면적 비율이 IQR 하한 미만",
                }
            )
        fig.suptitle("Red rectangle: candidate label (all candidates shown)")
        plt.tight_layout()
        plt.show()

# 경계 오류는 원래 사진과 좌표를 확장한 그림을 나란히 보여 준다.
# 사진 범위 밖의 bbox는 원래 사진 위에서는 보이지 않을 수 있다.
boundary_count = 0
for relative_json_path in quality_report.loc[
    quality_report["문제"] == "라벨 오류", "파일"
].unique():
    with (PROJECT_ROOT / relative_json_path).open(encoding="utf-8") as file:
        json_data = json.load(file)
    images_by_id = {image["id"]: image for image in json_data["images"]}
    for label in json_data["annotations"]:
        bbox = label.get("bbox")
        if (
            not isinstance(bbox, list)
            or len(bbox) != 4
            or not all(isinstance(value, (int, float)) and math.isfinite(value) for value in bbox)
        ):
            continue
        image_info = images_by_id.get(label.get("image_id"))
        if image_info is None:
            continue
        image_path = PROJECT_ROOT / "data" / "raw" / "train_images" / image_info["file_name"]
        if not image_path.is_file():
            continue
        left, top, width, height = bbox
        with Image.open(image_path) as image:
            image_width, image_height = image.size
            if (
                width <= 0
                or height <= 0
                or not (
                    left < 0 or top < 0 or left + width > image_width or top + height > image_height
                )
            ):
                continue
            boundary_count += 1
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            for ax in axes:
                ax.imshow(image.convert("RGB"))
                ax.add_patch(
                    Rectangle(
                        (0, 0), image_width, image_height, fill=False, color="lime", linewidth=2
                    )
                )
                ax.add_patch(
                    Rectangle((left, top), width, height, fill=False, color="red", linewidth=2)
                )
            axes[0].set(
                xlim=(0, image_width), ylim=(image_height, 0), title="Original image extent"
            )
            axes[1].set(
                xlim=(min(0, left) - 100, max(image_width, left + width) + 100),
                ylim=(max(image_height, top + height) + 100, min(0, top) - 100),
                title="Expanded coordinates: green=image, red=bbox",
            )
            fig.suptitle(f"Boundary error: class={label['category_id']}, bbox={bbox}")
            plt.tight_layout()
            plt.show()
        print(f"경계 오류 이미지: {image_info['file_name']}")
        print(f"이미지 범위: x=0~{image_width}, y=0~{image_height}")
        print(f"bbox 범위: x={left}~{left + width}, y={top}~{top + height}")

candidate_review = pd.DataFrame(review_rows)
print(f"통계적 후보 {len(review_rows)}개, 경계 오류 {boundary_count}개 시각화 완료")
if not candidate_review.empty:
    print("\n후보 번호별 판정 이유 (위 그림의 Candidate 번호와 대응):")
    print(candidate_review.to_string(index=False))
print("후보별 이유는 candidate_review에서 확인한다. 오류 좌표를 추정해 자동 수정하지 않는다.")

- 평균적인 면적 범위에 벗어나서(더 넓음) 이상치 후보로 등록된거같음
- 이미지 확인 결과 그냥 면적만 큰거같음 다른 이상이 없음
- 라벨 오류는 해결이 필요해보임

## 시각화 결과 해석

### 1. 좌표 오류 1개: 수정 검토가 필요한 라벨

- 이미지: `K-003351-016262-018357_0_2_0_2_75_000_200.png`
- 종류 ID: `18357`, bbox: `[6567, 625, 311, 315]`
- 사진 범위는 x=0~976, y=0~1280인데 박스의 x 범위가 6567~6878이므로 사진 밖에 있다.
- 왼쪽은 원본 사진, 오른쪽은 bbox까지 포함하도록 좌표를 확장한 그림이다. 초록색은 사진 경계, 빨간색은 라벨이다.
- 학습에 그대로 쓰면 잘못된 위치를 전달할 수 있다. 올바른 좌표를 사진·원본 자료와 대조한 후 수정하거나 해당 라벨의 제외를 검토한다. 6567의 의도된 값을 추측해서 바꾸지 않는다.

### 2. 면적 이상치 후보 24개: 오류 확정이 아닌 크기 검토 대상

- 면적 비율 = `(bbox 너비 × bbox 높이) / (이미지 너비 × 이미지 높이)`.
- 유효 bbox들의 면적 비율에서 Q1(25% 지점), Q3(75% 지점)을 구하고 `IQR = Q3 - Q1`로 계산한다.
- `Q1 - 1.5 × IQR`보다 작거나 `Q3 + 1.5 × IQR`보다 크면 후보로 표시한다. 이 기준은 정답의 허용 범위가 아니라 통계적 참고 기준이다.
- 현재 후보 24개는 모두 상한 약 **14.24%**보다 큰 bbox다. 그림의 `area`는 해당 bbox의 면적 비율이다.
- 빨간 박스가 큰 캡슐이나 정제를 감싸는 모습이므로, 시각 검토상 크기만으로 잘못된 라벨이라고 판단할 근거는 없다. 회전한 긴 알약은 축에 평행한 bbox 안에 배경이 더 들어갈 수 있다.
- 같은 종류의 알약이 크면 여러 사진이 함께 후보로 잡힐 수 있다. 종류별 크기 차이도 고려해야 한다.
- `Candidate 1`~`Candidate 24`를 아래 `candidate_review` 표의 후보 번호와 비교하면 사진·종류·면적 비율·선택 이유를 확인할 수 있다.

### 그림에서 확인할 것

1. 박스가 대상 알약 전체를 포함하는가?
2. 알약과 관계없는 위치나 다른 알약을 과도하게 포함하지 않는가?
3. 알약 일부가 박스 밖으로 잘려 있지 않은가?

여기서는 위치와 크기를 확인한다. 알약 종류의 정확성이나 다른 알약의 라벨 누락까지 검증한 것은 아니다. 원본 이미지와 JSON은 자동 수정하지 않는다.


## 경계 오류 1개 — OpenCV bbox·area 수정 후보
아래 셀만 실행하면 된다. 종근당글리아티린연질캡슐(클래스 18357), 라벨 903의 x=6567 오류를 검토한다.
잘못된 x를 추측해 줄이는 대신, 이 사진의 색상 차이로 윤곽을 찾고 기존 y·크기와 가까운 후보를 제안한다. y·크기는 맞는다는 가정이므로 시각 확인이 필요하다.
원본 사진·경계 밖 좌표·추출 마스크·새 박스를 제품명과 클래스 ID와 함께 비교한다. 새 area는 새 너비×높이로 계산한다.
메모리 복사본 boundary_repaired_data와 변경 근거 boundary_repair_record만 만든다. 노트북 외의 파일이나 원본은 수정하지 않는다.

In [ ]:
import copy
import hashlib
import json
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image

# 이 셀만 실행하면 된다. x=6567 경계 오류를 사진의 윤곽으로 재검토한다.
current_dir = Path.cwd().resolve()
boundary_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "data/raw/train_annotations").is_dir()
    ),
    None,
)
if boundary_root is None:
    raise FileNotFoundError("data/raw가 있는 프로젝트 안에서 실행해 주세요.")
boundary_source_path = (
    boundary_root
    / "data/raw/train_annotations/K-003351-016262-018357_json/K-018357"
    / "K-003351-016262-018357_0_2_0_2_75_000_200.json"
)
boundary_source_bytes = boundary_source_path.read_bytes()
boundary_original_data = json.loads(boundary_source_bytes)
targets = [
    label
    for label in boundary_original_data["annotations"]
    if label.get("id") == 903 and label.get("category_id") == 18357
]
if len(targets) != 1:
    raise ValueError("대상 라벨을 하나로 확정할 수 없습니다.")
boundary_original_label = targets[0]
if boundary_original_label["bbox"] != [6567, 625, 311, 315]:
    raise ValueError("원본 bbox가 이전 검사와 다릅니다. 변경된 라벨을 먼저 확인해 주세요.")
if boundary_original_label.get("segmentation"):
    raise ValueError("segmentation이 있어 분할 좌표도 함께 검토해야 합니다.")
boundary_image_info = next(
    image
    for image in boundary_original_data["images"]
    if image["id"] == boundary_original_label["image_id"]
)
boundary_image_path = boundary_root / "data/raw/train_images" / boundary_image_info["file_name"]
with Image.open(boundary_image_path) as image:
    boundary_rgb = np.array(image.convert("RGB"))
image_height, image_width = boundary_rgb.shape[:2]
old_left, old_top, old_width, old_height = boundary_original_label["bbox"]

# 푸른 배경과 갈색·노란 알약의 색상 차이로 후보를 추출한다. 이 사진에 한정된 설정이다.
hsv = cv2.cvtColor(boundary_rgb, cv2.COLOR_RGB2HSV)
boundary_mask = cv2.inRange(hsv, np.array([0, 45, 35]), np.array([45, 255, 255]))
kernel = np.ones((5, 5), dtype=np.uint8)
boundary_mask = cv2.morphologyEx(boundary_mask, cv2.MORPH_CLOSE, kernel)
boundary_mask = cv2.morphologyEx(boundary_mask, cv2.MORPH_OPEN, kernel)
contours, _ = cv2.findContours(boundary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
boundary_candidates = []
for contour in contours:
    if not 2000 < cv2.contourArea(contour) < image_width * image_height * 0.3:
        continue
    left, top, width, height = cv2.boundingRect(contour)
    # 잘못된 x는 사용하지 않는다. 남은 y·크기가 맞다는 가정으로 우선순위를 정한다.
    score = (
        abs(top - old_top) / image_height
        + abs(width - old_width) / image_width
        + abs(height - old_height) / image_height
    )
    padding = 8
    new_left, new_top = max(0, left - padding), max(0, top - padding)
    new_right = min(image_width, left + width + padding)
    new_bottom = min(image_height, top + height + padding)
    boundary_candidates.append(
        {
            "점수": score,
            "윤곽": contour,
            "bbox": [new_left, new_top, new_right - new_left, new_bottom - new_top],
        }
    )
boundary_candidates.sort(key=lambda candidate: candidate["점수"])
if not boundary_candidates:
    raise ValueError("윤곽 후보를 찾지 못했습니다. 수정안을 만들지 않습니다.")
if (
    len(boundary_candidates) > 1
    and boundary_candidates[1]["점수"] - boundary_candidates[0]["점수"] < 0.05
):
    raise ValueError("두 후보의 점수가 비슷합니다. 대상 알약을 먼저 확인해야 합니다.")
selected_candidate = boundary_candidates[0]
boundary_proposed_bbox = selected_candidate["bbox"]
new_left, new_top, new_width, new_height = boundary_proposed_bbox
if not (
    0 <= new_left < new_left + new_width <= image_width
    and 0 <= new_top < new_top + new_height <= image_height
):
    raise ValueError("새 박스가 이미지 범위를 벗어났습니다.")
boundary_proposed_area = new_width * new_height

# 원본 대신 메모리 복사본의 bbox·area만 함께 갱신한다. 다른 파일을 저장하지 않는다.
boundary_repaired_data = copy.deepcopy(boundary_original_data)
boundary_updated_label = next(
    label
    for label in boundary_repaired_data["annotations"]
    if label["id"] == 903 and label["image_id"] == boundary_original_label["image_id"]
)
boundary_updated_label["bbox"] = boundary_proposed_bbox
boundary_updated_label["area"] = boundary_proposed_area
boundary_repair_record = {
    "제품명": boundary_image_info.get("dl_name"),
    "클래스 ID": 18357,
    "라벨 ID": 903,
    "원본 JSON": boundary_source_path.relative_to(boundary_root).as_posix(),
    "원본 SHA256": hashlib.sha256(boundary_source_bytes).hexdigest(),
    "수정 전 bbox": boundary_original_label["bbox"],
    "수정 전 area": boundary_original_label["area"],
    "제안 bbox": boundary_proposed_bbox,
    "제안 area": boundary_proposed_area,
    "근거": "사진의 색상 윤곽과 기존 y·크기를 대조하여 경계 안의 후보 박스 재산정",
    "상태": "시각 확인용 후보·원본 미변경",
}

boundary_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        boundary_font = font_manager.FontProperties(
            fname=font_manager.findfont(font_name, fallback_to_default=False)
        )
        break
    except ValueError:
        continue
if boundary_font is None:
    font_path = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if font_path.is_file():
        boundary_font = font_manager.FontProperties(fname=font_path)

fig, axes = plt.subplots(2, 2, figsize=(13, 14))
for ax in axes.flat:
    ax.imshow(boundary_rgb)
    ax.set_xlim(0, image_width)
    ax.set_ylim(image_height, 0)
axes[0, 0].set_title("Original image: old bbox is outside this view")
axes[0, 1].add_patch(
    Rectangle((0, 0), image_width, image_height, fill=False, edgecolor="lime", linewidth=2)
)
axes[0, 1].add_patch(
    Rectangle((old_left, old_top), old_width, old_height, fill=False, edgecolor="red", linewidth=2)
)
axes[0, 1].set_xlim(0, max(image_width, old_left + old_width) + 150)
axes[0, 1].set_ylim(image_height, 0)
axes[0, 1].set_title(f"Old bbox: {boundary_original_label['bbox']} (red)", fontsize=10)
selected_mask = np.zeros(boundary_mask.shape, dtype=np.uint8)
cv2.drawContours(selected_mask, [selected_candidate["윤곽"]], -1, 1, thickness=cv2.FILLED)
axes[1, 0].imshow(
    np.ma.masked_where(selected_mask == 0, selected_mask), cmap="autumn", alpha=0.35, vmin=0, vmax=1
)
axes[1, 0].set_title("Estimated target mask: check capsule edges")
axes[1, 1].add_patch(
    Rectangle((new_left, new_top), new_width, new_height, fill=False, edgecolor="lime", linewidth=2)
)
axes[1, 1].text(
    new_left,
    new_top,
    "ID 18357",
    color="black",
    bbox={"facecolor": "lime", "alpha": 0.85, "pad": 2},
)
axes[1, 1].set_title(
    f"Proposal: {boundary_proposed_bbox}\narea=w*h={boundary_proposed_area}", fontsize=10
)
if boundary_font:
    fig.suptitle(
        f"경계 오류 1개 | {boundary_image_info.get('dl_name')} | 원본 클래스 ID: 18357",
        fontproperties=boundary_font,
        fontsize=14,
    )
else:
    fig.suptitle("Boundary error | Class ID: 18357")
plt.tight_layout()
plt.show()
plt.close(fig)
for number, candidate in enumerate(boundary_candidates, start=1):
    print(f"윤곽 후보 {number}: bbox={candidate['bbox']}, 비교 점수={candidate['점수']:.4f}")
print(json.dumps(boundary_repair_record, ensure_ascii=False, indent=2))
print("메모리 수정본: boundary_repaired_data / 수정 근거: boundary_repair_record")
print("대상 캡슐과 초록 박스를 대조하세요. y·크기도 틀린 경우 후보 선택이 틀릴 수 있습니다.")
print("수정본을 메모리에서만 만들었습니다. 원본 이미지·JSON과 다른 파일은 변경하지 않았습니다.")

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# 노트북 실행 위치에서 프로젝트 루트를 찾는다.
current_dir = Path.cwd().resolve()
project_root = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "pyproject.toml").is_file() and (path / "data").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

image_name = "K-003351-016262-018357_0_2_0_2_75_000_200.png"
image_path = project_root / "data/raw/train_images" / image_name
annotation_dir = project_root / "data/raw/train_annotations"

# 이미지의 기본 정보와 픽셀을 읽는다.
with Image.open(image_path) as image:
    image.load()
    image_width, image_height = image.size
    image_format = image.format
    image_mode = image.mode
    image_metadata = image.info.copy()
    image_exif = dict(image.getexif())
    display_image = image.convert("RGB")

print("========== 이미지 기본 정보 ==========")
print("파일명:", image_name)
print("경로:", image_path.relative_to(project_root))
print("파일 크기:", f"{image_path.stat().st_size:,} bytes")
print("이미지 형식:", image_format)
print("너비 × 높이:", f"{image_width} × {image_height} pixels")
print("색상 모드:", image_mode)
print("추가 메타데이터:", image_metadata)
print("EXIF:", image_exif)

# 같은 이미지의 라벨이 종류별 JSON에 나뉘어 있을 수 있어 모두 확인한다.
# 해당 이미지와 이름이 같은 JSON만 읽는다.
annotation_paths = sorted(annotation_dir.rglob(f"{Path(image_name).stem}.json"))
labels = []
matched_json_count = 0

for annotation_path in annotation_paths:
    with annotation_path.open(encoding="utf-8") as file:
        annotation_data = json.load(file)

    matching_images = {
        item["id"]: item
        for item in annotation_data.get("images", [])
        if item.get("file_name") == image_name
    }
    if not matching_images:
        continue

    matched_json_count += 1
    categories = {item["id"]: item for item in annotation_data.get("categories", [])}

    print("\n========== 연결된 JSON 전체 내용 ==========")
    print("경로:", annotation_path.relative_to(project_root))
    print(json.dumps(annotation_data, ensure_ascii=False, indent=2))

    for annotation in annotation_data.get("annotations", []):
        if annotation.get("image_id") not in matching_images:
            continue

        labels.append(
            {
                "annotation": annotation,
                "image_info": matching_images[annotation["image_id"]],
                "category": categories.get(annotation.get("category_id"), {}),
                "json_path": annotation_path.relative_to(project_root).as_posix(),
            }
        )

print("\n========== 라벨 요약 및 좌표 검사 ==========")
print("연결된 JSON 수:", matched_json_count)
print("연결된 라벨 수:", len(labels))

valid_boxes = []

for number, item in enumerate(labels, start=1):
    annotation = item["annotation"]
    image_info = item["image_info"]
    bbox = annotation.get("bbox")

    print(f"\n라벨 {number}")
    print("JSON:", item["json_path"])
    print("라벨 ID:", annotation.get("id"))
    print("종류 ID:", annotation.get("category_id"))
    print("종류 정보:", item["category"])
    print("제품명:", image_info.get("dl_name"))
    print("bbox [x, y, 너비, 높이]:", bbox)
    print("JSON의 area:", annotation.get("area"))
    print("segmentation:", annotation.get("segmentation"))
    print(
        "JSON과 실제 이미지 크기 일치:",
        (image_info.get("width"), image_info.get("height")) == (image_width, image_height),
    )

    if not isinstance(bbox, list) or len(bbox) != 4:
        print("오류: bbox 형식을 확인하세요.")
        continue

    left, top, width, height = bbox
    right = left + width
    bottom = top + height

    is_in_bounds = 0 <= left < right <= image_width and 0 <= top < bottom <= image_height

    print("왼쪽 위:", (left, top))
    print("오른쪽 아래:", (right, bottom))
    print("bbox 계산 면적:", width * height)
    print("이미지 경계 내에 있는가:", is_in_bounds)

    if not is_in_bounds:
        print("⚠ 경계 오류: 박스가 사진 밖에 있거나 크기가 올바르지 않습니다.")

    valid_boxes.append((number, annotation, is_in_bounds))

# 왼쪽은 원본, 오른쪽은 사진 밖의 bbox까지 포함한 좌표 공간이다.
fig, axes = plt.subplots(1, 2, figsize=(16, 9))

axes[0].imshow(display_image)
axes[0].set_title("Original image")
axes[0].set_xlabel("x (pixels)")
axes[0].set_ylabel("y (pixels)")

axes[1].imshow(display_image)
axes[1].add_patch(
    Rectangle(
        (0, 0),
        image_width,
        image_height,
        fill=False,
        edgecolor="lime",
        linewidth=2,
    )
)

x_values = [0, image_width]
y_values = [0, image_height]

for number, annotation, is_in_bounds in valid_boxes:
    left, top, width, height = annotation["bbox"]
    color = "dodgerblue" if is_in_bounds else "red"

    axes[1].add_patch(
        Rectangle(
            (left, top),
            width,
            height,
            fill=False,
            edgecolor=color,
            linewidth=2,
        )
    )
    axes[1].text(
        left,
        top,
        f"Label {number} / category {annotation.get('category_id')}",
        color=color,
        fontsize=10,
    )
    x_values.extend([left, left + width])
    y_values.extend([top, top + height])

axes[1].set_xlim(min(x_values) - 30, max(x_values) + 30)
axes[1].set_ylim(max(y_values) + 30, min(y_values) - 30)
axes[1].set_aspect("equal")
axes[1].set_title("All boxes / green: image boundary / red: invalid")
axes[1].set_xlabel("x (pixels)")
axes[1].set_ylabel("y (pixels)")

plt.tight_layout()
plt.show()

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# 프로젝트 루트를 찾는다.
current_dir = Path.cwd().resolve()
project_root = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "pyproject.toml").is_file() and (path / "data").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

image_names = [
    "K-003351-016262-018357_0_2_0_2_75_000_200.png",
    "K-003351-016262-018357_0_2_0_2_90_000_200.png",
]
titles = ["Boundary error image", "Comparison image"]
target_category_id = 18357

for title, image_name in zip(titles, image_names, strict=True):
    image_path = project_root / "data/raw/train_images" / image_name

    with Image.open(image_path) as image:
        image.load()
        image_width, image_height = image.size
        display_image = image.convert("RGB")

    # 해당 사진에 연결된 대상 종류의 라벨을 찾는다.
    targets = []
    annotation_paths = sorted(
        (project_root / "data/raw/train_annotations").rglob(f"{Path(image_name).stem}.json")
    )

    for annotation_path in annotation_paths:
        with annotation_path.open(encoding="utf-8") as file:
            annotation_data = json.load(file)

        matching_image_ids = {
            item["id"]
            for item in annotation_data.get("images", [])
            if item.get("file_name") == image_name
        }

        for annotation in annotation_data.get("annotations", []):
            if (
                annotation.get("image_id") in matching_image_ids
                and annotation.get("category_id") == target_category_id
            ):
                targets.append(annotation)

    print(f"\n이미지: {image_name}")
    print(f"이미지 크기: {image_width} × {image_height}")

    # 이미지마다 별도의 작은 그림을 만든다.
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.imshow(display_image)

    # 초록색 점선으로 실제 사진 경계를 표시한다.
    ax.add_patch(
        Rectangle(
            (0, 0),
            image_width,
            image_height,
            fill=False,
            edgecolor="green",
            linestyle="--",
            linewidth=1,
        )
    )

    x_values = [0, image_width]
    y_values = [0, image_height]

    if not targets:
        print("대상 라벨을 찾지 못했습니다.")

    for annotation in targets:
        left, top, width, height = annotation["bbox"]
        right = left + width
        bottom = top + height

        is_in_bounds = 0 <= left < right <= image_width and 0 <= top < bottom <= image_height
        color = "lime" if is_in_bounds else "red"

        print("종류 ID:", annotation["category_id"])
        print("bbox [x, y, 너비, 높이]:", annotation["bbox"])
        print("경계 검사:", "정상 범위" if is_in_bounds else "경계 오류")

        ax.add_patch(
            Rectangle(
                (left, top),
                width,
                height,
                fill=False,
                edgecolor=color,
                linewidth=1.5,
            )
        )
        ax.annotate(
            f"bbox: {annotation['bbox']}",
            xy=(left, top),
            xytext=(0, 6),
            textcoords="offset points",
            fontsize=7,
            color="black",
            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": color},
        )

        x_values.extend([left, right])
        y_values.extend([top, bottom])

    # 사진 밖의 bbox도 모두 포함한다.
    ax.set_xlim(min(x_values) - 100, max(x_values) + 100)
    ax.set_ylim(max(y_values) + 150, min(y_values) - 150)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x (pixels)", fontsize=8)
    ax.set_ylabel("y (pixels)", fontsize=8)
    ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.show()

- 같은 annotations 데이터 폴더 안에 있는 가장 비슷한 이미지와도 비교해봄. 다른 이미지의 bbox는 문제없음
- 혹시 데이터를 불러오고 압축 과정에서 오류가 생겼을 수도 있기 때문에 압축 전 원본 데이터도 직접 확인해봤음
- 압축을 풀고 불러온 데이터와 같음
- 결과 진짜 annotations 데이터가 잘못된거임

## JSON 필드 결측치 확인
앞의 이미지·라벨 연결 여부 검사와 달리 JSON 내부 값이 비었는지 확인한다. 이 셀은 단독 실행 가능하다.
문서의 필수/선택/조건부 항목을 구분하고 키 없음, null, 빈 문자열, 빈 배열을 집계한다.
bbox·종류 ID 등 핵심 필드와 부가 설명값을 구분한다. 빈 값이 해당 없음을 뜻할 수도 있으므로 일괄 삭제하지 않는다.
원본 라벨을 검사하며 수정본과 사진 픽셀은 읽거나 변경하지 않는다.

In [ ]:
import json
import math
from pathlib import Path

import pandas as pd

# 단독 실행할 수 있도록 프로젝트 루트를 찾는다.
current_dir = Path.cwd().resolve()
missing_project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file() and (folder / "data/README.md").is_file()
    ),
    None,
)
if missing_project_root is None:
    raise FileNotFoundError("프로젝트 안에서 노트북을 실행해 주세요.")

# 문서의 필드를 사용한다. M은 필수, O는 선택, C는 조건부다.
missing_schema = {}
for line in (missing_project_root / "data/README.md").read_text(encoding="utf-8").splitlines():
    if line.startswith("| `"):
        columns = [part.strip() for part in line.strip().strip("|").split("|")]
        if len(columns) >= 4 and columns[2] in {"M", "O", "C"}:
            missing_schema[columns[0].strip("`")] = columns[2]
if not missing_schema:
    raise ValueError("데이터 문서에서 필드 정의를 찾지 못했습니다.")

missing_details_rows = []
missing_checked_counts = {field: 0 for field in missing_schema}
missing_schema_errors = []
missing_json_count = 0
missing_annotation_dir = missing_project_root / "data/raw/train_annotations"
if not missing_annotation_dir.is_dir():
    raise FileNotFoundError(f"라벨 폴더가 없습니다: {missing_annotation_dir}")

for json_path in sorted(missing_annotation_dir.rglob("*")):
    if not json_path.is_file() or json_path.suffix.lower() != ".json":
        continue
    relative_path = json_path.relative_to(missing_project_root).as_posix()
    try:
        with json_path.open(encoding="utf-8") as file:
            json_data = json.load(file)
        if not isinstance(json_data, dict):
            raise ValueError("JSON 최상위는 객체여야 합니다.")
    except (OSError, UnicodeError, ValueError) as error:
        missing_schema_errors.append({"JSON": relative_path, "오류": str(error)})
        continue
    missing_json_count += 1
    for field_path, requirement in missing_schema.items():
        if "[]." in field_path:
            section, field = field_path.split("[].", 1)
            records = json_data.get(section, [])
            if records is None:
                continue
            if not isinstance(records, list):
                missing_schema_errors.append(
                    {"JSON": relative_path, "오류": f"{section}이 배열이 아님"}
                )
                continue
        else:
            section, field, records = "root", field_path, [json_data]
        for index, record in enumerate(records):
            if not isinstance(record, dict):
                missing_schema_errors.append(
                    {"JSON": relative_path, "오류": f"{section}[{index}]가 객체가 아님"}
                )
                continue
            missing_checked_counts[field_path] += 1
            value = record.get(field)
            kind = None
            if field not in record:
                kind = "키 없음"
            elif value is None:
                kind = "null"
            elif isinstance(value, str) and not value.strip():
                kind = "빈 문자열"
            elif isinstance(value, (list, dict)) and not value:
                kind = "빈 배열/객체"
            elif isinstance(value, float) and not math.isfinite(value):
                kind = "NaN/무한대"
            elif (
                field_path == "annotations[].bbox"
                and isinstance(value, list)
                and any(
                    part is None
                    or (isinstance(part, str) and not part.strip())
                    or (isinstance(part, float) and not math.isfinite(part))
                    for part in value
                )
            ):
                kind = "bbox 내부 결측"
            # 0과 False는 결측이 아니다. 타입과 좌표 범위 오류는 품질 검사에서 다룬다.
            if kind is not None:
                missing_details_rows.append(
                    {
                        "JSON": relative_path,
                        "항목": field_path,
                        "필수여부": requirement,
                        "레코드 인덱스": index,
                        "결측 유형": kind,
                        "이미지": record.get("file_name") if section == "images" else None,
                    }
                )

missing_details = pd.DataFrame(
    missing_details_rows,
    columns=["JSON", "항목", "필수여부", "레코드 인덱스", "결측 유형", "이미지"],
)
missing_summary_rows = []
for field_path, rows in missing_details.groupby("항목", sort=True):
    total = missing_checked_counts[field_path]
    missing_summary_rows.append(
        {
            "항목": field_path,
            "필수여부": missing_schema[field_path],
            "검사 수": total,
            "결측 수": len(rows),
            "결측률(%)": round(100 * len(rows) / total, 2),
            "유형별 수": rows["결측 유형"].value_counts().to_dict(),
        }
    )
missing_summary = pd.DataFrame(missing_summary_rows)
core_fields = {
    "images",
    "annotations",
    "categories",
    "images[].id",
    "images[].file_name",
    "images[].width",
    "images[].height",
    "annotations[].id",
    "annotations[].bbox",
    "annotations[].image_id",
    "annotations[].category_id",
    "categories[].id",
}
core_missing_values = missing_details[missing_details["항목"].isin(core_fields)]
null_metadata_images = (
    missing_details.loc[missing_details["결측 유형"] == "null", "이미지"].dropna().unique()
)

print(
    f"읽은 원본 JSON: {missing_json_count:,}개 / "
    f"파싱·구조 검사 기록: {len(missing_schema_errors)}건"
)
print(f"학습 핵심 필드 결측: {len(core_missing_values)}건")
print(f"null인 이미지 메타데이터가 있는 사진: {len(null_metadata_images)}장")
print("\n항목별 결측 (JSON별 레코드 기준, 동일 사진의 반복 메타데이터 포함):")
print(missing_summary.to_string(index=False) if not missing_summary.empty else "결측 없음")
if missing_schema_errors:
    print(pd.DataFrame(missing_schema_errors).drop_duplicates().to_string(index=False))
print("세부 파일: missing_details / 핵심 항목: core_missing_values / 항목별 집계: missing_summary")
print("빈 마크·분할선·선택 항목은 해당 없음일 수 있어 자동 오류 판정하지 않는다.")
print("원본 JSON만 검사한다. 테스트 라벨 미제공과 사진 속 알약의 라벨 누락은 필드 결측과 별개다.")

## 부가 설명이 null인 데이터 확인 — 이 셀만 실행 가능
앞의 결측 검사 셀을 실행하지 않아도 된다. 원본 JSON을 직접 읽어 부가 설명의 실제 null만 확인한다.
빈 문자열·키 없음과 이미지 ID·파일명·크기는 이 표에서 제외한다. 사진·제품명·null 항목·원본 JSON 경로를 표시한다.
이 코드 셀 다음에 맨 아래 이미지 전체 확인 셀을 실행하면 된다. 원본 데이터는 수정하지 않는다.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

# 이 셀만 실행해도 된다. 앞 셀의 missing_details 등은 사용하지 않는다.
current_dir = Path.cwd().resolve()
missing_project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "data/raw/train_annotations").is_dir()
    ),
    None,
)
if missing_project_root is None:
    raise FileNotFoundError("data/raw/train_annotations가 있는 프로젝트에서 실행해 주세요.")

null_core_image_fields = {"id", "file_name", "width", "height"}
null_metadata_rows = []
null_field_rows = []
null_read_errors = []
for json_path in sorted((missing_project_root / "data/raw/train_annotations").rglob("*.json")):
    json_name = json_path.relative_to(missing_project_root).as_posix()
    try:
        with json_path.open(encoding="utf-8") as file:
            json_data = json.load(file)
        if not isinstance(json_data, dict) or not isinstance(json_data.get("images"), list):
            raise ValueError("images 배열이 필요합니다.")
        for record_index, metadata in enumerate(json_data["images"]):
            if not isinstance(metadata, dict):
                raise ValueError(f"images[{record_index}]가 객체가 아닙니다.")
            # 키가 있으면서 값이 실제 null인 부가 설명만 고른다. 빈 문자열은 제외한다.
            fields = sorted(
                field
                for field, value in metadata.items()
                if field not in null_core_image_fields and value is None
            )
            if not fields:
                continue
            null_metadata_rows.append(
                {
                    "이미지": metadata.get("file_name"),
                    "제품명": metadata.get("dl_name"),
                    "알약코드": metadata.get("drug_N"),
                    "null 항목 수": len(fields),
                    "null 항목": ", ".join(fields),
                    "실제 저장값": {field: metadata[field] for field in fields},
                    "JSON": json_name,
                }
            )
            for field in fields:
                null_field_rows.append(
                    {
                        "JSON": json_name,
                        "레코드 인덱스": record_index,
                        "이미지": metadata.get("file_name"),
                        "항목": f"images[].{field}",
                        "결측 유형": "null",
                    }
                )
    except (OSError, UnicodeError, ValueError, TypeError) as error:
        null_read_errors.append({"JSON": json_name, "오류": str(error)})

null_metadata_fields = pd.DataFrame(
    null_field_rows, columns=["JSON", "레코드 인덱스", "이미지", "항목", "결측 유형"]
)

null_metadata_report = pd.DataFrame(
    null_metadata_rows,
    columns=["이미지", "제품명", "알약코드", "null 항목 수", "null 항목", "실제 저장값", "JSON"],
)
print(f"부가 설명에 null이 있는 사진: {null_metadata_report['이미지'].nunique()}장")
print(f"해당 이미지 메타데이터 레코드: {len(null_metadata_report)}개")
print(f"null 필드 값: {len(null_metadata_fields)}건")

if null_metadata_report.empty:
    print("부가 설명이 null인 레코드가 없습니다.")
else:
    # 모든 대상 사진과 항목을 생략 없이 보여 준다.
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(
            null_metadata_report[
                ["이미지", "제품명", "알약코드", "null 항목 수", "null 항목", "JSON"]
            ]
        )
        print("항목별 null 개수:")
        display(null_metadata_fields.groupby("항목").size().rename("null 수").to_frame())
    print("첫 대상의 실제 저장값 (JSON null은 파이썬에서 None):")
    print(null_metadata_report.iloc[0]["실제 저장값"])

print("전체 결과: null_metadata_report / 필드별 상세: null_metadata_fields")
print("같은 사진의 모든 알약 정보가 없다는 뜻은 아니다. 표의 제품명에 연결된 부가 설명만 해당한다.")
print("bbox·종류 ID 결측과 구분하며, 원본 JSON은 수정하지 않는다.")
if null_read_errors:
    print("일부 JSON은 읽지 못했으므로 결과가 불완전할 수 있습니다.")
    display(pd.DataFrame(null_read_errors))

## null 이미지 18장 — 제품명·원본 클래스 ID 함께 확인
부가 설명이 null인 데이터 확인 셀을 실행한 뒤 아래 셀을 실행한다. 6장씩 전체 사진을 표시한다.
사진 위에는 제품명과 원본 category_id, 빨간 박스에는 해당 클래스 ID를 표시한다. 파일명과 결측 항목은 사진 아래에도 출력한다.
클래스 ID는 JSON 원본 번호이며 모델의 예측값이나 학습용 연속 번호가 아니다. 원본 데이터는 변경하지 않는다.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image

# '부가 설명이 null인 데이터 확인' 셀을 먼저 실행한다.
if "null_metadata_report" not in globals():
    raise RuntimeError("먼저 '부가 설명이 null인 데이터 확인' 셀을 실행해 주세요.")

# 이미 설치된 한글 글꼴만 사용한다. 설치나 설정 파일 변경은 하지 않는다.
null_title_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        font_path = font_manager.findfont(font_name, fallback_to_default=False)
        null_title_font = font_manager.FontProperties(fname=font_path)
        break
    except ValueError:
        continue
if null_title_font is None:
    # WSL에서는 Windows에 있는 맑은 고딕을 읽어 사용할 수 있다.
    windows_font = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if windows_font.is_file():
        null_title_font = font_manager.FontProperties(fname=windows_font)
if null_title_font is None:
    print("한글 글꼴을 찾지 못해 그림에는 클래스 ID를, 아래 출력에는 제품명을 표시합니다.")

null_image_examples = null_metadata_report.drop_duplicates(["이미지", "JSON"])
null_image_label_rows = []
print(f"확인할 사진: {null_image_examples['이미지'].nunique()}장")
print("클래스 번호는 JSON의 원본 category_id입니다. 학습용 연속 번호로 변환한 값이 아닙니다.")

for start in range(0, len(null_image_examples), 6):
    page = null_image_examples.iloc[start : start + 6]
    page_descriptions = []
    fig, axes = plt.subplots(2, 3, figsize=(15, 14))
    for ax in axes.flat:
        ax.axis("off")

    for offset, (_, row) in enumerate(page.iterrows()):
        ax = axes.flat[offset]
        number = start + offset + 1
        image_path = missing_project_root / "data/raw/train_images" / row["이미지"]
        with (missing_project_root / row["JSON"]).open(encoding="utf-8") as file:
            image_json = json.load(file)
        image_ids = {
            image["id"]
            for image in image_json["images"]
            if image["file_name"] == row["이미지"] and image.get("drug_N") == row["알약코드"]
        }
        matched_labels = [
            label for label in image_json["annotations"] if label["image_id"] in image_ids
        ]
        category_names = {
            category["id"]: category.get("name", "이름 없음")
            for category in image_json.get("categories", [])
        }
        category_ids = sorted({label["category_id"] for label in matched_labels})
        class_text = ", ".join(map(str, category_ids)) or "No linked label"
        product_name = row["제품명"] or " / ".join(
            category_names.get(key, "이름 없음") for key in category_ids
        )

        with Image.open(image_path) as image:
            ax.imshow(image.convert("RGB"))
            image_width, image_height = image.size
        for label in matched_labels:
            left, top, width, height = label["bbox"]
            ax.add_patch(
                Rectangle((left, top), width, height, fill=False, edgecolor="red", linewidth=2)
            )
            ax.text(
                left,
                top,
                f"ID {label['category_id']}",
                color="white",
                fontsize=10,
                bbox={"facecolor": "red", "alpha": 0.8, "pad": 2},
            )
        ax.set_xlim(0, image_width)
        ax.set_ylim(image_height, 0)
        if null_title_font is not None:
            ax.set_title(
                f"[{number}] {product_name}\n원본 클래스 ID: {class_text}",
                fontproperties=null_title_font,
                fontsize=12,
            )
        else:
            ax.set_title(f"No. {number} | Class ID: {class_text}")
        description = {
            "번호": number,
            "이미지": row["이미지"],
            "제품명": product_name,
            "원본 클래스 ID": category_ids,
            "null 항목": row["null 항목"],
        }
        page_descriptions.append(description)
        null_image_label_rows.append(description)

    plt.tight_layout()
    plt.show()
    plt.close(fig)
    for description in page_descriptions:
        print(f"\n[{description['번호']}] {description['이미지']}")
        print("제품명:", description["제품명"])
        print("원본 클래스 ID:", description["원본 클래스 ID"])
        print("null 항목:", description["null 항목"])

if null_image_examples.empty:
    print("부가 설명이 null인 이미지가 없습니다.")
print("이름·클래스는 JSON 기록을 표시한 것이며 모델의 인식 결과가 아닙니다.")

## null 목록 4번 이미지 — 기록된 정답 박스 전체 확인
아래 셀은 단독 실행할 수 있다. 해당 사진을 참조하는 JSON을 모두 모아 제품명·클래스 ID·bbox를 표시한다.
동일 좌표에 붙은 서로 다른 종류는 색상·점선·번호로 구분한다. 사진 속 모든 알약이 올바르게 라벨링됐다고 보장하는 것은 아니다.
원본 이미지와 JSON은 변경하지 않는다.

In [ ]:
import json
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager
from matplotlib.patches import Patch, Rectangle
from PIL import Image

# null 이미지 목록의 4번 사진이다. 앞 셀을 실행하지 않아도 된다.
fourth_image_name = "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png"
current_dir = Path.cwd().resolve()
fourth_project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "data/raw/train_annotations").is_dir()
    ),
    None,
)
if fourth_project_root is None:
    raise FileNotFoundError("data/raw가 있는 프로젝트에서 실행해 주세요.")

# 파일명이 아니라 JSON 내부 file_name과 image_id로 사진의 모든 라벨을 찾는다.
fourth_rows = []
fourth_json_paths = []
for json_path in sorted((fourth_project_root / "data/raw/train_annotations").rglob("*.json")):
    with json_path.open(encoding="utf-8") as file:
        image_json = json.load(file)
    image_ids = {
        image["id"] for image in image_json["images"] if image["file_name"] == fourth_image_name
    }
    if not image_ids:
        continue
    fourth_json_paths.append(json_path)
    names = {
        category["id"]: category.get("name", "이름 없음") for category in image_json["categories"]
    }
    for label in image_json["annotations"]:
        if label["image_id"] in image_ids:
            fourth_rows.append(
                {
                    "번호": len(fourth_rows) + 1,
                    "제품명": names.get(label["category_id"], "이름 없음"),
                    "클래스 ID": label["category_id"],
                    "bbox": label["bbox"],
                    "라벨 ID": label.get("id"),
                    "JSON": json_path.relative_to(fourth_project_root).as_posix(),
                }
            )
if not fourth_rows:
    raise ValueError("4번 사진에 연결된 라벨을 찾지 못했습니다.")
fourth_box_report = pd.DataFrame(fourth_rows)

# 이미 설치된 한글 글꼴만 읽는다.
fourth_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        fourth_font = font_manager.FontProperties(
            fname=font_manager.findfont(font_name, fallback_to_default=False)
        )
        break
    except ValueError:
        continue
if fourth_font is None:
    font_path = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if font_path.is_file():
        fourth_font = font_manager.FontProperties(fname=font_path)

colors = ["red", "lime", "deepskyblue", "orange", "magenta", "yellow"]
same_box_counts = defaultdict(int)
legend_handles = []
image_path = fourth_project_root / "data/raw/train_images" / fourth_image_name
with Image.open(image_path) as image:
    fig, ax = plt.subplots(figsize=(10, 13))
    ax.imshow(image.convert("RGB"))
    image_width, image_height = image.size
for index, row in enumerate(fourth_rows):
    left, top, width, height = row["bbox"]
    color = colors[index % len(colors)]
    box_key = tuple(row["bbox"])
    overlap_index = same_box_counts[box_key]
    same_box_counts[box_key] += 1
    # 같은 좌표의 박스는 점선과 실선을 겹쳐 표시하고 텍스트를 다른 높이에 쓴다.
    ax.add_patch(
        Rectangle(
            (left, top),
            width,
            height,
            fill=False,
            edgecolor=color,
            linewidth=3,
            linestyle="--" if overlap_index else "-",
        )
    )
    ax.text(
        left,
        top + overlap_index * 42,
        f"{row['번호']}: ID {row['클래스 ID']}",
        color="black",
        fontsize=11,
        va="top",
        bbox={"facecolor": color, "alpha": 0.85, "pad": 2},
    )
    legend_text = (
        f"{row['번호']}. {row['제품명']} (ID {row['클래스 ID']})"
        if fourth_font
        else f"{row['번호']}. Class ID {row['클래스 ID']}"
    )
    legend_handles.append(Patch(facecolor="none", edgecolor=color, label=legend_text))
ax.set_xlim(0, image_width)
ax.set_ylim(image_height, 0)
ax.set_title("Image No. 4: all recorded labels (including conflicting boxes)")
ax.axis("off")
ax.legend(
    handles=legend_handles,
    prop=fourth_font,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=2,
)
plt.tight_layout()
plt.show()
plt.close(fig)

print(f"연결된 JSON: {len(fourth_json_paths)}개 / 라벨: {len(fourth_rows)}개")
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    display(fourth_box_report)
print("동일 bbox에 다른 종류가 붙은 경우:")
for box_key in same_box_counts:
    matches = [row for row in fourth_rows if tuple(row["bbox"]) == box_key]
    if len({row["클래스 ID"] for row in matches}) > 1:
        print("bbox:", list(box_key))
        print("충돌:", [(row["제품명"], row["클래스 ID"]) for row in matches])
print(
    "그림은 JSON에 기록된 라벨 전체를 보여 줍니다. 충돌 라벨까지 올바른 정답이라는 뜻은 아닙니다."
)

- json 부가설명이 null 알약들이 있음 총 18개
- null 이어도 18개 알약들의 정보를 확인하는데 문제가 없는건가?
- 18개중 17개는 이상없이 알약의 이름과 클래스 번호가 뜸
- 4번 이미지에서 알약의 이름이랑 클래스 번호가 다름
- 4번을 열어본 결과 bbox 가 겹처있는걸 확인
- 일단 json 부가설명이 null 이어도 알약의 정보를 확인하는데는 문제가 없음

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# 다른 셀 실행 없이 프로젝트 경로를 찾는다.
current_dir = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "data/raw/train_annotations").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("프로젝트 안에서 실행해 주세요.")

empty_material_samples = []

for json_path in sorted((project_root / "data/raw/train_annotations").rglob("*.json")):
    with json_path.open(encoding="utf-8") as file:
        data = json.load(file)

    for image_info in data["images"]:
        material = image_info.get("dl_material")

        # null이 아니라 빈 문자열 또는 공백 문자열만 선택한다.
        if not isinstance(material, str) or material.strip():
            continue

        labels = [label for label in data["annotations"] if label["image_id"] == image_info["id"]]
        empty_material_samples.append(
            {
                "이미지": image_info["file_name"],
                "제품명": image_info.get("dl_name"),
                "성분명": material,
                "라벨": labels,
                "JSON": json_path.relative_to(project_root).as_posix(),
            }
        )

print("성분명 빈 문자열 레코드:", len(empty_material_samples))
print("해당 사진 수:", len({row["이미지"] for row in empty_material_samples}))

# 해당 사진 전체를 6장씩 표시한다.
for start in range(0, len(empty_material_samples), 6):
    page = empty_material_samples[start : start + 6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 13))

    for ax in axes.flat:
        ax.axis("off")

    for offset, sample in enumerate(page):
        ax = axes.flat[offset]
        number = start + offset + 1

        with Image.open(project_root / "data/raw/train_images" / sample["이미지"]) as image:
            ax.imshow(image.convert("RGB"))
            image_width, image_height = image.size

        class_ids = []
        for label in sample["라벨"]:
            left, top, width, height = label["bbox"]
            class_ids.append(label["category_id"])

            ax.add_patch(
                Rectangle(
                    (left, top),
                    width,
                    height,
                    fill=False,
                    edgecolor="red",
                    linewidth=2,
                )
            )
            ax.text(
                left,
                top,
                f"ID {label['category_id']}",
                color="white",
                bbox={"facecolor": "red", "alpha": 0.8, "pad": 2},
            )

        ax.set_xlim(0, image_width)
        ax.set_ylim(image_height, 0)
        ax.set_title(f"No. {number} | Class ID: {class_ids}")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    for offset, sample in enumerate(page):
        print(f"\n[{start + offset + 1}] {sample['이미지']}")
        print("제품명:", sample["제품명"])
        print("성분명 실제 값:", repr(sample["성분명"]))
        print("JSON:", sample["JSON"])

## 성분명 있음·없음 이미지 비교 — 제품명과 클래스 ID 표시
아래 셀은 단독 실행할 수 있다. 성분명이 빈 9개 레코드를 모두 표시하며 왼쪽은 빈 문자열, 오른쪽은 성분명 값이 있는 비교 사례다. null은 이번 대상에서 제외한다.
같은 클래스의 값이 있는 사진을 우선 찾는다. 현재 빈 문자열 사례인 다보타민큐정(12778)·트루비타정(19552)은 같은 클래스의 성분명 기록이 없어 다른 클래스와 비교한다.
색상과 모양을 통해 성분을 판별하는 실험이 아니다. 사진 제목에 제품명·원본 클래스 ID·성분명을 함께 표시하고, 출력에서도 비교 대상과 파일 경로를 확인한다. 원본 데이터는 변경하지 않는다.

In [ ]:
import json
import os
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image

# 이 셀만 실행해도 된다. 성분명이 빈 문자열인 사례와 값이 있는 사례를 비교한다.
current_dir = Path.cwd().resolve()
material_project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "data/raw/train_annotations").is_dir()
    ),
    None,
)
if material_project_root is None:
    raise FileNotFoundError("data/raw가 있는 프로젝트 안에서 실행해 주세요.")

material_missing_samples = []
material_present_samples = []
for json_path in sorted((material_project_root / "data/raw/train_annotations").rglob("*.json")):
    with json_path.open(encoding="utf-8") as file:
        image_json = json.load(file)
    for metadata in image_json["images"]:
        material = metadata.get("dl_material")
        # null은 이번 빈 문자열 비교 대상과 구분한다.
        if not isinstance(material, str):
            continue
        labels = [
            label for label in image_json["annotations"] if label["image_id"] == metadata["id"]
        ]
        sample = {
            "이미지": metadata["file_name"],
            "제품명": metadata.get("dl_name", "이름 없음"),
            "클래스 ID": tuple(sorted({label["category_id"] for label in labels})),
            "성분명": material,
            "라벨": labels,
            "JSON": json_path.relative_to(material_project_root).as_posix(),
        }
        if material.strip():
            material_present_samples.append(sample)
        else:
            material_missing_samples.append(sample)

# 이미 설치된 한글 글꼴만 읽는다. 새 패키지나 글꼴은 설치하지 않는다.
material_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        material_font = font_manager.FontProperties(
            fname=font_manager.findfont(font_name, fallback_to_default=False)
        )
        break
    except ValueError:
        continue
if material_font is None:
    font_path = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if font_path.is_file():
        material_font = font_manager.FontProperties(fname=font_path)

# 같은 클래스의 기록된 성분명이 없으면, 다른 클래스 대표 사진을 골라 명시한다.
representatives = {}
for sample in material_present_samples:
    representatives.setdefault(sample["클래스 ID"], sample)
material_comparison_pairs = []
for number, missing in enumerate(material_missing_samples, start=1):
    comparable = [
        sample for sample in material_present_samples if sample["클래스 ID"] == missing["클래스 ID"]
    ]
    if comparable:
        present = comparable[0]
        comparison_reason = "같은 클래스 비교"
    else:
        alternatives = [
            sample for sample in representatives.values() if sample["이미지"] != missing["이미지"]
        ]
        if not alternatives:
            print(f"{missing['제품명']}: 비교할 기록 있음 샘플이 없습니다.")
            continue
        present = alternatives[(number - 1) % len(alternatives)]
        comparison_reason = "같은 클래스의 성분명 기록이 없어 다른 클래스와 비교"
    material_comparison_pairs.append((missing, present, comparison_reason))

print(f"성분명 빈 문자열: {len(material_missing_samples)}개 레코드")
print("왼쪽: 성분명 없음 / 오른쪽: 성분명 기록 있음")
print("값이 있다는 것은 해당 필드의 기록 여부만 뜻하며 전체 라벨의 정상 판정은 아닙니다.")

# 모든 빈 문자열 사례를 표시한다. 한 페이지에는 비교 3쌍을 넣는다.
for start in range(0, len(material_comparison_pairs), 3):
    page = material_comparison_pairs[start : start + 3]
    fig, axes = plt.subplots(len(page), 2, figsize=(14, 7 * len(page)), squeeze=False)
    for row_index, (missing, present, _reason) in enumerate(page):
        number = start + row_index + 1
        for column, sample in enumerate((missing, present)):
            ax = axes[row_index, column]
            with Image.open(
                material_project_root / "data/raw/train_images" / sample["이미지"]
            ) as image:
                ax.imshow(image.convert("RGB"))
                image_width, image_height = image.size
            color = "red" if column == 0 else "lime"
            for label in sample["라벨"]:
                left, top, width, height = label["bbox"]
                ax.add_patch(
                    Rectangle((left, top), width, height, fill=False, edgecolor=color, linewidth=2)
                )
                ax.text(
                    left,
                    top,
                    f"ID {label['category_id']}",
                    color="black",
                    fontsize=10,
                    bbox={"facecolor": color, "alpha": 0.85, "pad": 2},
                )
            class_text = ", ".join(map(str, sample["클래스 ID"]))
            state = "성분명 없음" if column == 0 else "성분명 기록 있음"
            if material_font:
                title = (
                    f"[{number}] {state}\n{sample['제품명']}\n원본 클래스 ID: {class_text}\n"
                    + textwrap.fill(f"성분명: {sample['성분명'] or '(빈 문자열)'}", width=30)
                )
                ax.set_title(title, fontproperties=material_font, fontsize=11)
            else:
                state = "Empty material" if column == 0 else "Material recorded"
                ax.set_title(f"Pair {number}: {state}\nClass ID: {class_text}")
            ax.set_xlim(0, image_width)
            ax.set_ylim(image_height, 0)
            ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    for offset, (missing, present, reason) in enumerate(page):
        print(f"\n비교 {start + offset + 1}: {reason}")
        for side, sample in (("성분명 없음", missing), ("성분명 기록 있음", present)):
            print(f"  {side}: {sample['제품명']} / 클래스 ID {sample['클래스 ID']}")
            print("  성분명:", repr(sample["성분명"]))
            print("  이미지:", sample["이미지"])
            print("  JSON:", sample["JSON"])

if material_font is None:
    print("한글 글꼴이 없어 제품명은 위의 출력에서 확인하세요.")
print("제품명·클래스 ID는 JSON 기록이며 모델의 예측값이 아닙니다. 원본은 수정하지 않았습니다.")

## area와 bbox 면적 불일치 이미지 시각화
아래 셀만 실행하면 된다. raw 원본에서 |area − bbox 너비×높이| > 1픽셀²인 라벨을 모두 표시한다.
사진마다 제품명·원본 클래스 ID·기록된 area·계산한 bbox 면적·차이와 차이율을 표시한다. 빨간 박스는 해당 원본 라벨이다.
같은 사진에 불일치 라벨이 2개면 사진을 각각 표시하므로 라벨 수와 고유 사진 수가 다를 수 있다.
면적 불일치만으로 bbox가 잘못됐다고 확정하지 않는다. area가 분할 영역 면적인지 bbox 면적인지 원본 정의를 확인해야 한다. 원본은 수정하지 않는다.

In [ ]:
import json
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image

# 이 셀만 실행한다. bbox 면적과 기록된 area가 1픽셀²보다 크게 다른 경우를 찾는다.
area_tolerance = 1.0
current_dir = Path.cwd().resolve()
area_project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "data/raw/train_annotations").is_dir()
    ),
    None,
)
if area_project_root is None:
    raise FileNotFoundError("data/raw가 있는 프로젝트 안에서 실행해 주세요.")

area_mismatch_samples = []
area_invalid_count = 0
for json_path in sorted((area_project_root / "data/raw/train_annotations").rglob("*.json")):
    with json_path.open(encoding="utf-8") as file:
        image_json = json.load(file)
    images_by_id = {image["id"]: image for image in image_json["images"]}
    category_names = {
        category["id"]: category.get("name", "이름 없음") for category in image_json["categories"]
    }
    for label in image_json["annotations"]:
        bbox = label.get("bbox")
        area = label.get("area")
        if (
            not isinstance(bbox, list)
            or len(bbox) != 4
            or any(
                isinstance(value, bool)
                or not isinstance(value, (int, float))
                or not math.isfinite(value)
                for value in bbox
            )
            or isinstance(area, bool)
            or not isinstance(area, (int, float))
            or not math.isfinite(area)
            or bbox[2] <= 0
            or bbox[3] <= 0
        ):
            area_invalid_count += 1
            continue
        bbox_area = bbox[2] * bbox[3]
        difference = area - bbox_area
        if abs(difference) <= area_tolerance:
            continue
        image_info = images_by_id[label["image_id"]]
        area_mismatch_samples.append(
            {
                "제품명": category_names.get(label["category_id"], "이름 없음"),
                "클래스 ID": label["category_id"],
                "라벨 ID": label.get("id"),
                "이미지": image_info["file_name"],
                "bbox": bbox,
                "기록된 area": area,
                "bbox 면적": bbox_area,
                "차이": difference,
                "차이율(%)": round(100 * difference / bbox_area, 2),
                "JSON": json_path.relative_to(area_project_root).as_posix(),
            }
        )

# 이미 설치된 글꼴만 사용해 제품명과 클래스 번호를 함께 표시한다.
area_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        area_font = font_manager.FontProperties(
            fname=font_manager.findfont(font_name, fallback_to_default=False)
        )
        break
    except ValueError:
        continue
if area_font is None:
    font_path = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if font_path.is_file():
        area_font = font_manager.FontProperties(fname=font_path)

print(f"면적 불일치 라벨: {len(area_mismatch_samples)}개")
print(f"해당 사진: {len({sample['이미지'] for sample in area_mismatch_samples})}장")
print(f"값 결측·형식·너비/높이 오류로 비교하지 못한 라벨: {area_invalid_count}개")
for start in range(0, len(area_mismatch_samples), 6):
    page = area_mismatch_samples[start : start + 6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 16))
    for ax in axes.flat:
        ax.axis("off")
    for offset, sample in enumerate(page):
        ax = axes.flat[offset]
        number = start + offset + 1
        with Image.open(area_project_root / "data/raw/train_images" / sample["이미지"]) as image:
            ax.imshow(image.convert("RGB"))
            image_width, image_height = image.size
        left, top, width, height = sample["bbox"]
        ax.add_patch(
            Rectangle((left, top), width, height, fill=False, edgecolor="red", linewidth=2)
        )
        ax.text(
            left,
            top,
            f"ID {sample['클래스 ID']}",
            color="white",
            fontsize=10,
            bbox={"facecolor": "red", "alpha": 0.8, "pad": 2},
        )
        ax.set_xlim(0, image_width)
        ax.set_ylim(image_height, 0)
        numbers = (
            f"area={sample['기록된 area']:g} | bbox={width:g} x {height:g}"
            f" = {sample['bbox 면적']:g}\n"
            f"difference={sample['차이']:+g} ({sample['차이율(%)']:+.2f}%)"
        )
        if area_font:
            ax.set_title(
                f"[{number}] {sample['제품명']}\n원본 클래스 ID: {sample['클래스 ID']}\n" + numbers,
                fontproperties=area_font,
                fontsize=10,
            )
        else:
            ax.set_title(f"No. {number} | Class ID: {sample['클래스 ID']}\n" + numbers, fontsize=10)
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    for offset, sample in enumerate(page):
        print(f"\n[{start + offset + 1}] {sample['이미지']}")
        print("제품명:", sample["제품명"], "/ 원본 클래스 ID:", sample["클래스 ID"])
        print("bbox:", sample["bbox"], "/ JSON:", sample["JSON"])

area_mismatch_report = pd.DataFrame(area_mismatch_samples)
if area_mismatch_report.empty:
    print("허용 오차를 넘는 면적 불일치가 없습니다.")
else:
    with pd.option_context(
        "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
    ):
        display(area_mismatch_report)
print("단위는 픽셀²이며 차이=기록된 area-bbox 너비×높이입니다.")
print("area가 객체/분할 영역 면적으로 정의됐다면 bbox 면적과 달라도 정상일 수 있습니다.")
print("원본 정의와 박스 위치를 확인하기 전에는 area나 bbox를 자동 수정하지 않습니다.")

## 면적 불일치 2번 — 사진 기준 bbox·area 수정 후보
콜리네이트연질캡슐 400mg(클래스 19232), 라벨 5178을 확인한다. 기존 bbox [523, 201, 232, 241]은 캡슐 아래쪽을 자르므로 area만 바꾸지 않는다.
이 사진용 색상 분리로 캡슐 윤곽을 찾고 경계 여유를 포함해 bbox를 재산정한다. 새 area는 새 bbox의 너비×높이로 함께 계산한다.
기존 박스·추출 마스크·수정 후보를 확대 비교한다. 반투명 가장자리가 누락될 수 있으므로 제안은 시각 확인이 필요하다. 기존 area에 맞춰 박스를 키우는 방식이 아니다.
area를 bbox 면적으로 사용하는 수정 후보이며 실제 분할 면적과는 다르다. 메모리 복사본만 만들고 원본 이미지·JSON이나 다른 파일은 변경하지 않는다.

In [ ]:
import copy
import hashlib
import json
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.patches import Rectangle
from PIL import Image

# 면적 불일치 목록 2번을 사진의 캡슐 경계 기준으로 검토한다. 이 셀만 실행 가능하다.
current_dir = Path.cwd().resolve()
area_repair_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file()
        and (folder / "data/raw/train_annotations").is_dir()
    ),
    None,
)
if area_repair_root is None:
    raise FileNotFoundError("data/raw가 있는 프로젝트 안에서 실행해 주세요.")
area_repair_path = (
    area_repair_root
    / "data/raw/train_annotations/K-003351-019232-029667_json/K-019232"
    / "K-003351-019232-029667_0_2_1_2_70_000_200.json"
)
source_bytes = area_repair_path.read_bytes()
original_data = json.loads(source_bytes)
targets = [
    label
    for label in original_data["annotations"]
    if label.get("id") == 5178 and label.get("category_id") == 19232
]
if len(targets) != 1:
    raise ValueError("수정 대상 라벨을 하나로 확정할 수 없습니다.")
original_label = targets[0]
if original_label["bbox"] != [523, 201, 232, 241] or original_label["area"] != 79112:
    raise ValueError("원본 값이 이전 검사와 다릅니다. 현재 사진과 JSON을 다시 확인하세요.")
if original_label.get("segmentation"):
    raise ValueError("segmentation이 있으므로 면적 정의와 분할 좌표를 함께 검토해야 합니다.")
image_info = next(
    image for image in original_data["images"] if image["id"] == original_label["image_id"]
)
image_path = area_repair_root / "data/raw/train_images" / image_info["file_name"]
with Image.open(image_path) as image:
    rgb = np.array(image.convert("RGB"))
image_height, image_width = rgb.shape[:2]

# 이 사진의 푸른 배경과 갈색·노란 캡슐을 분리한다. 다른 사진용 범용 검출기가 아니다.
hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
color_mask = cv2.inRange(hsv, np.array([0, 45, 35]), np.array([45, 255, 255]))
kernel = np.ones((5, 5), dtype=np.uint8)
color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_CLOSE, kernel)
color_mask = cv2.morphologyEx(color_mask, cv2.MORPH_OPEN, kernel)
_, components, stats, _ = cv2.connectedComponentsWithStats(color_mask)
old_left, old_top, old_width, old_height = original_label["bbox"]
center_x, center_y = old_left + old_width // 2, old_top + old_height // 2
component_id = int(components[center_y, center_x])
if component_id == 0:
    raise ValueError("원본 박스 중앙에서 캡슐 윤곽을 찾지 못했습니다. 자동 제안을 중단합니다.")
mask_left, mask_top, mask_width, mask_height, foreground_pixels = map(int, stats[component_id])
target_mask = components == component_id

# 반투명 경계가 마스크에서 빠질 수 있어 8픽셀 여유를 두고 사진으로 확인한다.
padding = 8
new_left, new_top = max(0, mask_left - padding), max(0, mask_top - padding)
new_right = min(image_width, mask_left + mask_width + padding)
new_bottom = min(image_height, mask_top + mask_height + padding)
proposed_bbox = [new_left, new_top, new_right - new_left, new_bottom - new_top]
proposed_area = proposed_bbox[2] * proposed_bbox[3]
if not (0 <= new_left < new_right <= image_width and 0 <= new_top < new_bottom <= image_height):
    raise ValueError("제안 bbox가 이미지 범위를 벗어났습니다.")

# 메모리의 복사본에서 라벨 하나의 bbox와 area만 함께 갱신한다. 파일 저장은 하지 않는다.
area_repaired_data = copy.deepcopy(original_data)
updated_label = next(
    label
    for label in area_repaired_data["annotations"]
    if label["id"] == 5178 and label["image_id"] == original_label["image_id"]
)
updated_label["bbox"] = proposed_bbox
updated_label["area"] = proposed_area
area_repair_record = {
    "원본 JSON": area_repair_path.relative_to(area_repair_root).as_posix(),
    "원본 SHA256": hashlib.sha256(source_bytes).hexdigest(),
    "제품명": image_info.get("dl_name"),
    "클래스 ID": 19232,
    "라벨 ID": 5178,
    "수정 전 bbox": original_label["bbox"],
    "수정 전 area": original_label["area"],
    "제안 bbox": proposed_bbox,
    "제안 area": proposed_area,
    "근거": "원본 bbox가 캡슐 아래쪽을 자름. 색상 윤곽과 여유 픽셀로 bbox 후보 재산정",
    "상태": "시각 확인용 후보·원본 미변경",
}

repair_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        repair_font = font_manager.FontProperties(
            fname=font_manager.findfont(font_name, fallback_to_default=False)
        )
        break
    except ValueError:
        continue
if repair_font is None:
    font_path = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if font_path.is_file():
        repair_font = font_manager.FontProperties(fname=font_path)

fig, axes = plt.subplots(1, 3, figsize=(17, 8))
for ax in axes:
    ax.imshow(rgb)
    # 캡슐 주변만 확대해 아래쪽이 잘리는지 쉽게 비교한다.
    ax.set_xlim(
        max(0, min(old_left, new_left) - 55),
        min(image_width, max(old_left + old_width, new_right) + 55),
    )
    ax.set_ylim(
        min(image_height, max(old_top + old_height, new_bottom) + 55),
        max(0, min(old_top, new_top) - 55),
    )
    ax.axis("off")
axes[0].add_patch(
    Rectangle((old_left, old_top), old_width, old_height, fill=False, edgecolor="red", linewidth=2)
)
axes[0].set_title(
    f"Before: {original_label['bbox']}\n"
    f"area={original_label['area']}, w*h={old_width * old_height}",
    fontsize=10,
)
axes[1].imshow(
    np.ma.masked_where(~target_mask, target_mask.astype(float)),
    cmap="autumn",
    alpha=0.35,
    vmin=0,
    vmax=1,
)
axes[1].set_title("Estimated capsule mask (check translucent edges)", fontsize=10)
axes[2].add_patch(
    Rectangle(
        (new_left, new_top),
        proposed_bbox[2],
        proposed_bbox[3],
        fill=False,
        edgecolor="lime",
        linewidth=2,
    )
)
axes[2].set_title(f"Proposal: {proposed_bbox}\narea=w*h={proposed_area}", fontsize=10)
if repair_font:
    fig.suptitle(
        f"면적 불일치 2번 | {image_info.get('dl_name')} | 원본 클래스 ID: 19232",
        fontproperties=repair_font,
        fontsize=14,
    )
else:
    fig.suptitle("Area mismatch No. 2 | Class ID: 19232")
plt.tight_layout()
plt.show()
plt.close(fig)
print(json.dumps(area_repair_record, ensure_ascii=False, indent=2))
print("메모리 수정본: area_repaired_data / 수정 근거: area_repair_record")
print("초록 박스가 반투명 캡슐 전체를 감싸는지 확인하세요. 색상 분리 결과는 정답 확정이 아닙니다.")
print("합의한 노트북만 수정 원칙에 따라 JSON 파일은 저장하거나 덮어쓰지 않았습니다.")

## 반복 촬영 후보 78그룹 확인 — 전체 목록과 그룹별 사진 비교
아래 셀은 단독 실행 가능하다. 파일명 첫 _ 앞의 알약 조합이 같고 사진이 2장 이상인 그룹을 모두 집계한다.
전체 목록은 capture_group_report, 사진 경로와 클래스 상세는 capture_group_details에서 확인한다.
**기본 그림은 1~5번 그룹이다.** 코드 위쪽 group_numbers_to_show를 [6, 7, 8]처럼 바꾸면 해당 그룹을 보고, None으로 바꾸면 78그룹 전체를 그린다.
사진과 함께 제품명·원본 클래스 ID·camera_la를 표시한다. 동일 원본의 증강인지 같은 조합의 재촬영인지는 구분해서 해석해야 한다. 원본은 변경하지 않는다.

In [ ]:
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager
from PIL import Image

# 이 셀만 실행하면 된다. 표에는 반복 조합 후보 전체를 표시한다.
# 그림으로 볼 그룹 번호를 고른다. None으로 바꾸면 전체 그룹 사진을 표시한다.
group_numbers_to_show = [1, 2, 3, 4, 5]

current_dir = Path.cwd().resolve()
group_project_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file() and (folder / "data/raw/train_images").is_dir()
    ),
    None,
)
if group_project_root is None:
    raise FileNotFoundError("data/raw가 있는 프로젝트에서 실행해 주세요.")
group_image_dir = group_project_root / "data/raw/train_images"
group_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
all_capture_groups = defaultdict(list)
for image_path in sorted(group_image_dir.rglob("*")):
    if image_path.is_file() and image_path.suffix.lower() in group_extensions:
        # 파일명 첫 _ 앞부분이 같은 사진을 같은 알약 조합 후보로 묶는다.
        all_capture_groups[image_path.stem.split("_", 1)[0]].append(image_path)
repeated_capture_groups = {
    key: paths for key, paths in sorted(all_capture_groups.items()) if len(paths) > 1
}
single_capture_groups = {key: paths for key, paths in all_capture_groups.items() if len(paths) == 1}

# 사진별 촬영 정보와 실제 JSON에 기록된 제품명·클래스 ID를 모은다.
group_photo_metadata = defaultdict(list)
group_photo_classes = defaultdict(dict)
group_json_errors = []
for json_path in sorted((group_project_root / "data/raw/train_annotations").rglob("*.json")):
    try:
        with json_path.open(encoding="utf-8") as file:
            image_json = json.load(file)
        images_by_id = {image["id"]: image for image in image_json["images"]}
        category_names = {
            category["id"]: category.get("name", "이름 없음")
            for category in image_json["categories"]
        }
        for image in image_json["images"]:
            group_photo_metadata[image["file_name"]].append(
                {
                    "camera_la": image.get("camera_la"),
                    "camera_lo": image.get("camera_lo"),
                    "조명": image.get("light_color"),
                }
            )
        for label in image_json["annotations"]:
            name = images_by_id[label["image_id"]]["file_name"]
            group_photo_classes[name][label["category_id"]] = category_names.get(
                label["category_id"], "이름 없음"
            )
    except (OSError, UnicodeError, ValueError, KeyError, TypeError) as error:
        group_json_errors.append(
            {"JSON": json_path.relative_to(group_project_root).as_posix(), "문제": str(error)}
        )

capture_group_details = {}
capture_group_rows = []
for group_number, (combination, image_paths) in enumerate(repeated_capture_groups.items(), start=1):
    product_classes = {}
    camera_angles = set()
    for image_path in image_paths:
        name = image_path.relative_to(group_image_dir).as_posix()
        product_classes.update(group_photo_classes[name])
        camera_angles.update(str(row["camera_la"]) for row in group_photo_metadata[name])
    capture_group_details[group_number] = {
        "조합": combination,
        "사진 경로": image_paths,
        "제품명·클래스": product_classes,
    }
    capture_group_rows.append(
        {
            "그룹 번호": group_number,
            "파일명 조합": combination,
            "이미지 수": len(image_paths),
            "camera_la 목록": sorted(camera_angles),
            "클래스 ID": sorted(product_classes),
            "제품명": list(product_classes.values()),
        }
    )
capture_group_report = pd.DataFrame(
    capture_group_rows,
    columns=["그룹 번호", "파일명 조합", "이미지 수", "camera_la 목록", "클래스 ID", "제품명"],
)
print(f"반복 조합 후보: {len(repeated_capture_groups)}그룹")
print(f"포함 사진: {sum(map(len, repeated_capture_groups.values()))}장")
print("그룹별 사진 수 분포:", dict(Counter(map(len, repeated_capture_groups.values()))))
print(f"한 장뿐인 조합: {len(single_capture_groups)}그룹")
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    display(capture_group_report)
if group_json_errors:
    print("일부 JSON을 읽지 못해 제품명·클래스 목록이 불완전할 수 있습니다.")
    display(pd.DataFrame(group_json_errors))

# 설치된 한글 글꼴만 사용하며 다른 파일이나 패키지는 수정하지 않는다.
group_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        group_font = font_manager.FontProperties(
            fname=font_manager.findfont(font_name, fallback_to_default=False)
        )
        break
    except ValueError:
        continue
if group_font is None:
    font_path = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if font_path.is_file():
        group_font = font_manager.FontProperties(fname=font_path)
selected_group_numbers = (
    list(capture_group_details) if group_numbers_to_show is None else group_numbers_to_show
)
for group_number in selected_group_numbers:
    if group_number not in capture_group_details:
        print(f"없는 그룹 번호: {group_number}")
        continue
    group = capture_group_details[group_number]
    paths = group["사진 경로"]
    fig, axes = plt.subplots(1, len(paths), figsize=(5 * len(paths), 10), squeeze=False)
    for image_number, (ax, image_path) in enumerate(zip(axes.flat, paths, strict=True), start=1):
        name = image_path.relative_to(group_image_dir).as_posix()
        with Image.open(image_path) as image:
            ax.imshow(image.convert("RGB"))
        angle_values = sorted({str(row["camera_la"]) for row in group_photo_metadata[name]})
        classes = sorted(group_photo_classes[name])
        ax.set_title(
            f"Group {group_number} / Image {image_number}\n"
            f"camera_la={','.join(angle_values)}\nClass IDs: {classes}",
            fontsize=9,
        )
        ax.axis("off")
    products = group["제품명·클래스"]
    if group_font:
        fig.suptitle(
            f"그룹 {group_number} | {group['조합']}", fontproperties=group_font, fontsize=12
        )
        product_text = "\n".join(
            f"{name} | 원본 클래스 ID: {category_id}"
            for category_id, name in sorted(products.items())
        )
        fig.text(
            0.5,
            0.02,
            product_text,
            ha="center",
            va="bottom",
            fontproperties=group_font,
            fontsize=11,
        )
    else:
        fig.suptitle(f"Group {group_number} | {group['조합']}")
        fig.text(0.5, 0.02, f"Class IDs: {sorted(products)}", ha="center")
    plt.tight_layout(rect=(0, 0.16, 1, 0.95))
    plt.show()
    plt.close(fig)
    print(f"\n그룹 {group_number}: {group['조합']}")
    for category_id, product_name in sorted(products.items()):
        print(f"  제품명: {product_name} / 원본 클래스 ID: {category_id}")
    for image_path in paths:
        print("  사진:", image_path.name)

print("표는 전체 후보, 그림은 group_numbers_to_show에서 선택한 그룹입니다.")
print(
    "같은 알약 조합의 후보입니다. 디지털 증강인지 반복 촬영인지는 이 검사만으로 확정하지 않습니다."
)
print("유사 사진이 train/val에 섞이는 것을 피하려면 촬영 조합 기준 그룹 분할을 검토하세요.")

## 한 장뿐인 알약 조합 36그룹 전체 시각화
아래 셀은 단독 실행한다. 파일명 첫 _ 앞부분을 조합으로 묶어, 현재 학습 폴더에서 같은 조합의 사진이 정확히 1장인 경우를 찾는다.
알약 한 종류가 한 장뿐이라는 뜻은 아니다. 같은 알약이 다른 조합의 사진에 다시 등장할 수 있다.
전체 목록은 single_group_report에 저장하며 사진 전체를 6장씩 표시한다. 사진마다 JSON 제품명·원본 클래스 ID·camera_la를 함께 표시한다.
이 정보만으로 데이터 유실이나 이상치로 판정하지 않는다. 원본 파일이나 다른 파일은 수정하지 않는다.

In [ ]:
import json
import os
import textwrap
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager
from PIL import Image

# 이 셀만 실행하면 된다. 같은 파일명 조합의 사진이 정확히 1장인 경우를 찾는다.
current_dir = Path.cwd().resolve()
single_group_root = next(
    (
        folder
        for folder in (current_dir, *current_dir.parents)
        if (folder / "pyproject.toml").is_file() and (folder / "data/raw/train_images").is_dir()
    ),
    None,
)
if single_group_root is None:
    raise FileNotFoundError("data/raw가 있는 프로젝트 안에서 실행해 주세요.")
single_image_dir = single_group_root / "data/raw/train_images"
single_extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
single_all_groups = defaultdict(list)
for image_path in sorted(single_image_dir.rglob("*")):
    if image_path.is_file() and image_path.suffix.lower() in single_extensions:
        single_all_groups[image_path.stem.split("_", 1)[0]].append(image_path)
single_group_paths = [
    (key, paths[0]) for key, paths in sorted(single_all_groups.items()) if len(paths) == 1
]
single_names = {
    image_path.relative_to(single_image_dir).as_posix() for key, image_path in single_group_paths
}
single_products = defaultdict(dict)
single_cameras = defaultdict(set)
single_jsons = defaultdict(set)
for json_path in sorted((single_group_root / "data/raw/train_annotations").rglob("*.json")):
    with json_path.open(encoding="utf-8") as file:
        image_json = json.load(file)
    selected_images = {
        image["id"]: image for image in image_json["images"] if image["file_name"] in single_names
    }
    if not selected_images:
        continue
    category_names = {
        category["id"]: category.get("name", "이름 없음") for category in image_json["categories"]
    }
    for image in selected_images.values():
        single_cameras[image["file_name"]].add(str(image.get("camera_la")))
        single_jsons[image["file_name"]].add(json_path.relative_to(single_group_root).as_posix())
    for label in image_json["annotations"]:
        if label["image_id"] in selected_images:
            name = selected_images[label["image_id"]]["file_name"]
            single_products[name][label["category_id"]] = category_names.get(
                label["category_id"], "이름 없음"
            )

single_group_rows = []
for number, (combination, image_path) in enumerate(single_group_paths, start=1):
    name = image_path.relative_to(single_image_dir).as_posix()
    products = single_products[name]
    single_group_rows.append(
        {
            "번호": number,
            "파일명 조합": combination,
            "이미지": name,
            "클래스 ID": sorted(products),
            "제품명": list(products.values()),
            "camera_la": sorted(single_cameras[name]),
            "연결 JSON 수": len(single_jsons[name]),
        }
    )
single_group_report = pd.DataFrame(
    single_group_rows,
    columns=["번호", "파일명 조합", "이미지", "클래스 ID", "제품명", "camera_la", "연결 JSON 수"],
)
print(f"한 장뿐인 조합: {len(single_group_report)}그룹 / {len(single_group_report)}장")
print("알약 한 종류가 한 장뿐이라는 뜻이 아닙니다. 이 조합의 사진 파일 수가 1이라는 뜻입니다.")
with pd.option_context(
    "display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None
):
    display(single_group_report)

single_font = None
for font_name in ("Malgun Gothic", "NanumGothic", "Noto Sans CJK KR", "AppleGothic"):
    try:
        single_font = font_manager.FontProperties(
            fname=font_manager.findfont(font_name, fallback_to_default=False)
        )
        break
    except ValueError:
        continue
if single_font is None:
    font_path = Path(os.environ.get("WINDIR", "/mnt/c/Windows")) / "Fonts/malgun.ttf"
    if font_path.is_file():
        single_font = font_manager.FontProperties(fname=font_path)

# 36장 전체를 6장씩 표시한다. 페이지마다 제품명과 클래스 ID를 함께 보여 준다.
for start in range(0, len(single_group_rows), 6):
    page = single_group_rows[start : start + 6]
    fig, axes = plt.subplots(2, 3, figsize=(18, 22))
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, page, strict=False):
        with Image.open(single_image_dir / row["이미지"]) as image:
            ax.imshow(image.convert("RGB"))
        products = single_products[row["이미지"]]
        if single_font:
            product_lines = "\n".join(
                textwrap.fill(f"{name} | ID {category_id}", width=32)
                for category_id, name in sorted(products.items())
            )
            title = (
                f"[{row['번호']}] 한 장뿐인 조합\ncamera_la={row['camera_la']}\n" + product_lines
            )
            ax.set_title(title, fontproperties=single_font, fontsize=9)
        else:
            ax.set_title(
                f"No. {row['번호']} | Single-image combination\n"
                f"Class IDs: {row['클래스 ID']}\ncamera_la={row['camera_la']}",
                fontsize=9,
            )
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    for row in page:
        print(f"\n[{row['번호']}] {row['이미지']}")
        for category_id, name in sorted(single_products[row["이미지"]].items()):
            print(f"  제품명: {name} / 원본 클래스 ID: {category_id}")
        print("  camera_la:", row["camera_la"], "/ 연결 JSON 수:", row["연결 JSON 수"])

if not single_group_rows:
    print("한 장뿐인 조합이 없습니다.")
print("현재 raw/train_images 안에서만 센 결과입니다. 다른 원본 파일의 유실 여부는 알 수 없습니다.")
print(
    "한 장뿐이라는 이유로 삭제하지 않습니다. 분할 시 종류별 train/val 포함 여부를 함께 확인하세요."
)
print("제품명과 클래스는 JSON 기록이며 실제 종류와 일치하는지는 별도 검토가 필요합니다.")

In [ ]:
# bbox 여백 검토 후보 집계 (자동 추정, 파일 저장 없음)

import json
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from IPython.display import display

# 프로젝트 루트 또는 notebooks 폴더에서 실행할 수 있다.
PROJECT_ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "data/raw/train_annotations").is_dir()
)


def measure_padding(img: np.ndarray, bbox: list[float]) -> dict:
    """배경과 다른 최대 성분의 최소 사각형으로 여백 비율을 추정한다."""
    left, top, width, height = map(int, bbox)
    img_height, img_width = img.shape[:2]
    if min(width, height) <= 0 or min(left, top) < 0:
        return {"status": "invalid_bbox"}
    if left + width > img_width or top + height > img_height:
        return {"status": "invalid_bbox"}
    margin = max(10, int(max(width, height) * 0.15))
    crop_left, crop_top = max(0, left - margin), max(0, top - margin)
    crop = img[crop_top : top + height + margin, crop_left : left + width + margin]
    scale = min(1.0, 300 / max(crop.shape[:2]))
    crop = cv2.resize(crop, None, fx=scale, fy=scale)
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(float)
    # 명도는 그림자의 영향을 받으므로 색도 두 채널로 배경을 분리한다.
    border = np.concatenate(
        (
            lab[:3].reshape(-1, 3),
            lab[-3:].reshape(-1, 3),
            lab[:, :3].reshape(-1, 3),
            lab[:, -3:].reshape(-1, 3),
        )
    )
    background = np.median(border[:, 1:], axis=0)
    noise = np.linalg.norm(border[:, 1:] - background, axis=1)
    threshold = max(12.0, float(np.median(noise) + 6 * np.median(abs(noise - np.median(noise)))))
    mask = (np.linalg.norm(lab[:, :, 1:] - background, axis=2) > threshold).astype("uint8")
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    count, components, stats, centers = cv2.connectedComponentsWithStats(mask)
    center = np.array(
        [(left + width / 2 - crop_left) * scale, (top + height / 2 - crop_top) * scale]
    )
    options = [
        i
        for i in range(1, count)
        if stats[i, 4] > width * height * scale**2 * 0.08
        and np.linalg.norm((centers[i] - center) / [width * scale, height * scale]) < 0.45
    ]
    if not options:
        return {"status": "segmentation_failed"}
    target = max(options, key=lambda i: stats[i, 4])
    contours, _ = cv2.findContours(
        (components == target).astype("uint8"), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
    tight = [crop_left + x / scale, crop_top + y / scale, w / scale, h / scale]
    # 원래 상자 밖으로 나온 윤곽은 잘림 후보이므로 여백 집계에서 분리한다.
    tolerance = 3 / scale
    if (
        tight[0] < left - tolerance
        or tight[1] < top - tolerance
        or tight[0] + tight[2] > left + width + tolerance
        or tight[1] + tight[3] > top + height + tolerance
    ):
        return {"status": "extends_outside", "tight_bbox": tight}
    return {
        "status": "estimated",
        "tight_bbox": tight,
        "padding_ratio": max(0.0, 1 - tight[2] * tight[3] / (width * height)),
    }


def count_bbox_padding() -> tuple[pd.DataFrame, dict]:
    """원본 대신 수정본을 우선 사용하고 이미지별로 한 번씩 읽어 집계한다."""
    cv2.setNumThreads(1)
    grouped = defaultdict(list)
    annotation_root = PROJECT_ROOT / "data/raw/train_annotations"
    corrected_root = PROJECT_ROOT / "data/processed/corrected_annotations"
    for source in sorted(annotation_root.rglob("*.json")):
        corrected = corrected_root / source.relative_to(annotation_root)
        data = json.loads((corrected if corrected.exists() else source).read_text())
        images = {item["id"]: item for item in data["images"]}
        for annotation in data["annotations"]:
            grouped[images[annotation["image_id"]]["file_name"]].append((source, annotation))
    rows = []
    for index, (name, annotations) in enumerate(grouped.items(), 1):
        img = cv2.imread(str(PROJECT_ROOT / "data/raw/train_images" / name))
        for source, annotation in annotations:
            result = (
                measure_padding(img, annotation["bbox"])
                if img is not None
                else {"status": "missing_image"}
            )
            rows.append(
                {
                    "image": name,
                    "source": str(source.relative_to(PROJECT_ROOT)),
                    "annotation_id": annotation["id"],
                    "category_id": annotation["category_id"],
                    "bbox": annotation["bbox"],
                    **result,
                }
            )
        if index % 50 == 0:
            print(f"진행: {index}/{len(grouped)} 이미지", flush=True)
    summary = {
        "images": len(grouped),
        "annotations": len(rows),
        "statuses": dict(Counter(row["status"] for row in rows)),
        "thresholds": {},
    }
    for threshold in (0.1, 0.2, 0.3):
        selected = [row for row in rows if row.get("padding_ratio", -1) >= threshold]
        summary["thresholds"][str(threshold)] = {
            "annotations": len(selected),
            "images": len({row["image"] for row in selected}),
            "categories": len({row["category_id"] for row in selected}),
        }
    return pd.DataFrame(rows), summary


# 파일 저장이나 원본 라벨 수정 없이 메모리에서만 집계한다.
bbox_padding_measurements, bbox_padding_summary = count_bbox_padding()
bbox_padding_candidates = bbox_padding_measurements.loc[
    bbox_padding_measurements["padding_ratio"].ge(0.2)
].sort_values("padding_ratio", ascending=False)
print(
    f"검사: 사진 {bbox_padding_summary['images']}장, bbox {bbox_padding_summary['annotations']}개"
)
print("판별 상태:", bbox_padding_summary["statuses"])
display(pd.DataFrame(bbox_padding_summary["thresholds"]).T.rename_axis("여백 비율 기준"))
print("20% 이상 여백 추정 후보 (오류 확정 아님):", len(bbox_padding_candidates))
display(bbox_padding_candidates)

In [ ]:
# 앞의 여백 집계 셀을 실행한 뒤 실행한다. 각 구간의 하한에 가장 가까운 예시를 고른다.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

if "bbox_padding_measurements" not in globals():
    raise RuntimeError("바로 위의 bbox 여백 집계 셀을 먼저 실행해 주세요.")

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
bbox_padding_examples = []
for ax, lower, upper in zip(axes, (0.1, 0.2, 0.3), (0.2, 0.3, 1.01), strict=True):
    candidates = bbox_padding_measurements.loc[
        bbox_padding_measurements["status"].eq("estimated")
        & bbox_padding_measurements["padding_ratio"].ge(lower)
        & bbox_padding_measurements["padding_ratio"].lt(upper)
    ].sort_values(["padding_ratio", "image", "annotation_id"])
    if candidates.empty:
        ax.set_title(f"{lower:.0%}: no example")
        ax.axis("off")
        continue
    example = candidates.iloc[0]
    bbox_padding_examples.append(example)
    img_path = PROJECT_ROOT / "data/raw/train_images" / example["image"]
    img = cv2.imread(str(img_path))
    if img is None:
        raise FileNotFoundError(img_path)
    left, top, width, height = example["bbox"]
    tight_left, tight_top, tight_width, tight_height = example["tight_bbox"]
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.add_patch(
        Rectangle(
            (left, top),
            width,
            height,
            fill=False,
            edgecolor="red",
            linewidth=2,
            label="Original bbox",
        )
    )
    ax.add_patch(
        Rectangle(
            (tight_left, tight_top),
            tight_width,
            tight_height,
            fill=False,
            edgecolor="lime",
            linewidth=2,
            label="Estimated tight bbox",
        )
    )
    # 알약 주변을 확대하되 원본 좌표를 유지한다.
    margin = max(width, height) * 0.15
    ax.set_xlim(max(0, left - margin), min(img.shape[1], left + width + margin))
    ax.set_ylim(min(img.shape[0], top + height + margin), max(0, top - margin))
    ax.set_title(
        f"{lower:.0%}+ example: {example['padding_ratio']:.1%}\n"
        f"annotation {example['annotation_id']} / class {example['category_id']}"
    )
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("빨강: 기존 bbox / 초록: 추정 윤곽의 최소 사각형")
print("10~20%, 20~30%, 30% 이상 구간에서 각각 한 개씩 선택했습니다. 자동 추정 예시입니다.")
for example in bbox_padding_examples:
    print(f"{example['padding_ratio']:.1%}: {example['image']} (라벨 {example['annotation_id']})")